# Анализ и расширение датасета Uzbek NER

Цель: изучить данные из `ner_uz_hackathon_participant`, оценить совместимость внешних NER-датасетов и подобрать источники gazetteer для классов `ORG`, `NAME` и `GEO`.

## План

### 1. Описать текущий датасет

- определить размеры `train/dev`;
- посчитать распределение `ORG`, `NAME`, `GEO`;
- исследовать длины текстов в символах и в токенах `mmBERT-base` и `XLM-R-base`;
- оценить доли латиницы, кириллицы и смешанных текстов;
- найти пустые записи, дубли и ошибки в разметке.

### 2. Проанализировать стилистику и возможное происхождение данных

- выделить новостные тексты, посты из социальных сетей, комментарии, рекламу и объявления;
- исследовать URL, упоминания аккаунтов, хэштеги, emoji, подписи источников и повторяющиеся шаблоны;
- определить тематические группы и различия их разметки;
- проверить метаданные, даты, характерные формулировки и совпадения с открытыми корпусами;
- сформулировать гипотезу о возможных источниках данных, явно отделяя подтверждённые факты от предположений.

### 3. Проверить покрытие сущностей

- определить, сколько сущностей из `dev` встречается в `train`;
- выяснить, какие классы содержат больше новых сущностей;
- вывести частые и редкие названия;
- сравнить точное совпадение с совпадением после casefold и нормализации апострофов.

### 4. Исследовать внешние NER-датасеты

- найти 2–4 подходящих источника;
- сравнить их формат с текущим JSONL;
- описать преобразование классов и BIO-разметки в символьные `start/end`;
- проверить язык, алфавит, домен, качество, лицензию и возможные пересечения с текущими данными.

### 5. Найти gazetteer-источники

- `GEO`: GeoNames и OpenStreetMap;
- `ORG/NAME/GEO`: Wikidata;
- `ORG`: официальные реестры организаций Узбекистана;
- для каждого источника указать формат, лицензию, покрытие, ограничения и возможную пользу.

### 6. Исследовать генерацию синтетических данных

- определить недостаточно представленные классы, стили, алфавиты и сложные случаи;
- рассмотреть замену сущностей в реальных шаблонах, шаблонную генерацию и генерацию с помощью LLM;
- генерировать варианты транслитерации, апострофов, регистра, опечаток и узбекских суффиксов;
- автоматически проверять метки, точные границы, отсутствие пересечений и дублей;
- хранить синтетику отдельно и измерять её влияние отдельным экспериментом.

### 7. Дать рекомендации

- какие данные стоит добавлять в первую очередь;
- какие источники требуют ручной проверки;
- какие источники лучше не использовать;
- как подключить gazetteer к модели;
- какой способ генерации синтетики безопаснее и полезнее для этой разметки.

## Подготовка окружения

Следующая ячейка устанавливает только отсутствующие библиотеки. Повторный запуск ничего не переустанавливает.

In [33]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas==2.3.3",
    "transformers": "transformers==5.14.1",
    "sentencepiece": "sentencepiece==0.2.1",
}

missing_packages = [
    package_spec
    for import_name, package_spec in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    raise RuntimeError(
        "Не хватает библиотек: " + ", ".join(missing_packages)
        + ". Запустите Jupyter через uv по инструкции docs/DATA_ANALYSIS.md."
    )
else:
    print("Все необходимые библиотеки уже установлены.")


Все необходимые библиотеки уже установлены.


In [34]:
from pathlib import Path
from collections import Counter
import json
import math
import unicodedata

import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "ner_uz_hackathon_participant" / "data"
TRAIN_PATH = DATA_DIR / "train.jsonl"
DEV_PATH = DATA_DIR / "dev.jsonl"

TRAIN_PATH, DEV_PATH

(PosixPath('/home/mag/itmo/ai_hack/ner_uz_hackathon_participant/data/train.jsonl'),
 PosixPath('/home/mag/itmo/ai_hack/ner_uz_hackathon_participant/data/dev.jsonl'))

## 1. Описание текущего датасета

Каждая строка файлов `train.jsonl` и `dev.jsonl` должна быть отдельным JSON-объектом с полями `hash`, `text` и `entities`. Для каждой сущности проверим метку и символьные границы `[start, end)`. Исходный текст не нормализуем, потому что координаты сущностей относятся именно к нему.

In [35]:
def read_jsonl(path: Path) -> list[dict]:
    """Читает JSONL и сообщает номер некорректной строки."""
    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                raise ValueError(f"{path}:{line_number}: пустая строка")
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"{path}:{line_number}: некорректный JSON: {error}"
                ) from error
    return records


train_records = read_jsonl(TRAIN_PATH)
dev_records = read_jsonl(DEV_PATH)

print(f"train: {len(train_records):,} документов")
print(f"dev:   {len(dev_records):,} документов")

train: 13,000 документов
dev:   1,500 документов


In [36]:
ALLOWED_LABELS = {"ORG", "NAME", "GEO"}
REQUIRED_RECORD_FIELDS = {"hash", "text", "entities"}
REQUIRED_ENTITY_FIELDS = {"label", "start", "end"}


def validate_records(records: list[dict], split_name: str) -> dict:
    """Проверяет схему, hash, метки, offsets, дубли и пересечения."""
    error_counts = Counter()
    error_examples = []
    seen_hashes = set()
    entity_count = 0

    def add_error(kind: str, message: str) -> None:
        error_counts[kind] += 1
        if len(error_examples) < 10:
            error_examples.append(message)

    for record_number, record in enumerate(records, start=1):
        location = f"{split_name}, запись {record_number}"

        if not isinstance(record, dict):
            add_error("record_not_object", f"{location}: запись не является объектом")
            continue

        missing_fields = REQUIRED_RECORD_FIELDS - set(record)
        if missing_fields:
            add_error("missing_record_fields", f"{location}: нет полей {sorted(missing_fields)}")
            continue

        record_hash = record["hash"]
        text = record["text"]
        entities = record["entities"]

        if not isinstance(record_hash, str) or not record_hash:
            add_error("invalid_hash", f"{location}: hash должен быть непустой строкой")
        elif record_hash in seen_hashes:
            add_error("duplicate_hash", f"{location}: повторяющийся hash {record_hash!r}")
        else:
            seen_hashes.add(record_hash)

        if not isinstance(text, str):
            add_error("invalid_text", f"{location}: text должен быть строкой")
            continue
        if not isinstance(entities, list):
            add_error("invalid_entities", f"{location}: entities должен быть списком")
            continue

        valid_spans = []
        seen_entities = set()
        for entity_number, entity in enumerate(entities, start=1):
            entity_count += 1
            entity_location = f"{location}, сущность {entity_number}"
            if not isinstance(entity, dict):
                add_error("entity_not_object", f"{entity_location}: не является объектом")
                continue

            missing_entity_fields = REQUIRED_ENTITY_FIELDS - set(entity)
            if missing_entity_fields:
                add_error("missing_entity_fields", f"{entity_location}: нет полей {sorted(missing_entity_fields)}")
                continue

            label = entity["label"]
            start = entity["start"]
            end = entity["end"]

            if label not in ALLOWED_LABELS:
                add_error("invalid_label", f"{entity_location}: неизвестная метка {label!r}")

            offsets_are_int = (
                isinstance(start, int) and not isinstance(start, bool)
                and isinstance(end, int) and not isinstance(end, bool)
            )
            if not offsets_are_int or not 0 <= start < end <= len(text):
                add_error("invalid_offsets", f"{entity_location}: неверные границы [{start}, {end})")
                continue

            entity_key = (label, start, end)
            if entity_key in seen_entities:
                add_error("duplicate_entity", f"{entity_location}: повтор сущности {entity_key}")
            seen_entities.add(entity_key)
            valid_spans.append((start, end, label))

        valid_spans.sort()
        for previous, current in zip(valid_spans, valid_spans[1:]):
            if previous[1] > current[0]:
                add_error("overlapping_entities", f"{location}: пересекаются {previous} и {current}")

    return {
        "split": split_name,
        "documents": len(records),
        "entities": entity_count,
        "unique_hashes": len(seen_hashes),
        "errors": sum(error_counts.values()),
        "errors_by_type": dict(error_counts),
        "error_examples": error_examples,
    }

In [37]:
validation_reports = [
    validate_records(train_records, "train"),
    validate_records(dev_records, "dev"),
]

for report in validation_reports:
    status = "OK" if report["errors"] == 0 else "НАЙДЕНЫ ОШИБКИ"
    print(
        f'{report["split"]}: {status}; '
        f'{report["documents"]:,} документов; '
        f'{report["entities"]:,} сущностей; '
        f'{report["unique_hashes"]:,} уникальных hash'
    )
    if report["errors_by_type"]:
        print("  Ошибки по типам:", report["errors_by_type"])
        print("  Примеры:", *report["error_examples"], sep="\n    - " )

train_hashes = {record["hash"] for record in train_records}
dev_hashes = {record["hash"] for record in dev_records}
print(f"Совпадающих hash между train и dev: {len(train_hashes & dev_hashes)}")

train: OK; 13,000 документов; 66,083 сущностей; 13,000 уникальных hash
dev: OK; 1,500 документов; 7,698 сущностей; 1,500 уникальных hash
Совпадающих hash между train и dev: 0


In [38]:
def preview_record(record: dict, max_text_length: int = 300) -> None:
    """Показывает текст и реальные подстроки размеченных сущностей."""
    text = record["text"]
    preview = text if len(text) <= max_text_length else text[:max_text_length] + "…"
    print("hash:", record["hash"])
    print("text:", preview)
    print("entities:")
    for entity in record["entities"]:
        mention = text[entity["start"]:entity["end"]]
        print(
            f'  {entity["label"]:<4} '
            f'[{entity["start"]}:{entity["end"]}] {mention!r}'
        )


preview_record(train_records[0])

hash: 0000e9b18d8c83339a9ecd4c8aa1000020260228
text: Тошкент вилоятининг Бўка-Бекобод йўлида ИИБ ходими томонидан уриб юборилиб, Дўстлик каналига ташлангани айтилган 43 ёшли фуқаронинг жасади 80 кун ўтиб топилди. Марҳумнинг яқинлари маълум қилишича, жасад 28-февраль куни соат 10:30 ларда Дўстлик каналидан, сув оқимининг қуйи қисмидан аниқланган. У қум…
entities:
  GEO  [0:19] 'Тошкент вилоятининг'
  GEO  [20:39] 'Бўка-Бекобод йўлида'
  ORG  [40:43] 'ИИБ'
  GEO  [76:92] 'Дўстлик каналига'
  GEO  [236:253] 'Дўстлик каналидан'
  ORG  [362:365] 'ИИБ'
  ORG  [493:509] 'вилоят ҳокимлиги'
  ORG  [584:587] 'ФВВ'
  ORG  [842:845] 'ИИБ'
  GEO  [959:975] 'Дўстлик каналига'


### 1.1. Размеры выборок и распределение классов

In [39]:
splits = {"train": train_records, "dev": dev_records}

dataset_rows = []
class_rows = []
for split_name, records in splits.items():
    label_counts = Counter(
        entity["label"]
        for record in records
        for entity in record["entities"]
    )
    total_entities = sum(label_counts.values())
    dataset_rows.append({
        "split": split_name,
        "documents": len(records),
        "with_entities": sum(bool(record["entities"]) for record in records),
        "without_entities": sum(not record["entities"] for record in records),
        "entities": total_entities,
    })
    for label in sorted(ALLOWED_LABELS):
        class_rows.append({
            "split": split_name,
            "label": label,
            "count": label_counts[label],
            "share, %": round(100 * label_counts[label] / total_entities, 2),
        })

dataset_summary = pd.DataFrame(dataset_rows).set_index("split")
class_distribution = pd.DataFrame(class_rows).set_index(["split", "label"])

display(dataset_summary)
display(class_distribution)

,documents,with_entities,without_entities,entities
split,,,,
train,13000,10614,2386,66083
dev,1500,1225,275,7698


count  share, %
split label                 
train GEO    21445     32.45
      NAME   21218     32.11
      ORG    23420     35.44
dev   GEO     2720     35.33
      NAME    2319     30.12
      ORG     2659     34.54

### 1.2. Длины текстов в символах

Для тяжёлохвостого распределения одной средней недостаточно, поэтому выводим медиану, несколько перцентилей и максимум.

In [40]:
def summarize_lengths(values: list[int]) -> dict:
    series = pd.Series(values, dtype="int64")
    return {
        "mean": round(series.mean(), 1),
        "median": int(series.median()),
        "p90": math.ceil(series.quantile(0.90)),
        "p95": math.ceil(series.quantile(0.95)),
        "p99": math.ceil(series.quantile(0.99)),
        "max": int(series.max()),
    }


character_length_rows = []
for split_name, records in splits.items():
    summary = summarize_lengths([len(record["text"]) for record in records])
    character_length_rows.append({"split": split_name, **summary})

character_length_summary = pd.DataFrame(character_length_rows).set_index("split")
character_length_summary

,mean,median,p90,p95,p99,max
split,,,,,,
train,497.4,151,1090,1865,4855,22668
dev,474.1,170,999,1719,4253,18975


### 1.3. Длины текстов в токенах mmBERT-base и XLM-R-base

Используются только токенизаторы, веса моделей не загружаются. В длину включены специальные токены, то есть результат можно сопоставлять с ограничением длины входа модели. При первом запуске токенизаторы будут скачаны с Hugging Face.

Если библиотека отсутствует, установите её в отдельной ячейке командой `%pip install transformers sentencepiece` и перезапустите kernel.

In [41]:
try:
    from transformers import AutoTokenizer
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "Не установлен transformers. Выполните `%pip install transformers sentencepiece`, "
        "перезапустите kernel и снова запустите notebook."
    ) from error

TOKENIZER_MODELS = {
    "mmBERT-base": "jhu-clsp/mmBERT-base",
    "XLM-R-base": "FacebookAI/xlm-roberta-base",
}


def token_lengths(texts: list[str], tokenizer, batch_size: int = 256) -> list[int]:
    """Считает длины без усечения и не хранит все токены одновременно."""
    lengths = []
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = 10**9  # отключаем предупреждения для длинных текстов
    try:
        for batch_start in range(0, len(texts), batch_size):
            batch = texts[batch_start:batch_start + batch_size]
            encoded = tokenizer(
                batch,
                add_special_tokens=True,
                truncation=False,
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False,
            )
            lengths.extend(len(input_ids) for input_ids in encoded["input_ids"])
    finally:
        tokenizer.model_max_length = original_max_length
    return lengths


token_length_rows = []
token_lengths_by_model = {}
for model_name, model_id in TOKENIZER_MODELS.items():
    print(f"Загрузка токенизатора {model_name}: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    token_lengths_by_model[model_name] = {}
    for split_name, records in splits.items():
        lengths = token_lengths([record["text"] for record in records], tokenizer)
        token_lengths_by_model[model_name][split_name] = lengths
        token_length_rows.append({
            "model": model_name,
            "split": split_name,
            **summarize_lengths(lengths),
        })

token_length_summary = pd.DataFrame(token_length_rows).set_index(["model", "split"])
token_length_summary

Загрузка токенизатора mmBERT-base: jhu-clsp/mmBERT-base
Загрузка токенизатора XLM-R-base: FacebookAI/xlm-roberta-base


mean  median  p90  p95   p99   max
model       split                                     
mmBERT-base train  197.2      65  435  752  1912  8573
            dev    187.6      70  409  689  1574  6810
XLM-R-base  train  159.6      55  348  590  1576  6769
            dev    151.8      60  327  537  1361  5415

### 1.4. Латиница, кириллица и смешанные тексты

Текст считаем смешанным, если не менее 5% его латинских и кириллических букв относятся к каждой из двух письменностей. Единичное английское название внутри кириллической статьи поэтому не делает весь документ смешанным.

In [42]:
SCRIPT_MIX_THRESHOLD = 0.05


def detect_script(text: str, mix_threshold: float = SCRIPT_MIX_THRESHOLD) -> str:
    latin = 0
    cyrillic = 0
    for character in text:
        if not character.isalpha():
            continue
        unicode_name = unicodedata.name(character, "")
        latin += "LATIN" in unicode_name
        cyrillic += "CYRILLIC" in unicode_name

    relevant_letters = latin + cyrillic
    if relevant_letters == 0:
        return "other"
    if (
        latin / relevant_letters >= mix_threshold
        and cyrillic / relevant_letters >= mix_threshold
    ):
        return "mixed"
    return "latin" if latin > cyrillic else "cyrillic"


script_rows = []
for split_name, records in splits.items():
    counts = Counter(detect_script(record["text"]) for record in records)
    for script in ("latin", "cyrillic", "mixed", "other"):
        script_rows.append({
            "split": split_name,
            "script": script,
            "documents": counts[script],
            "share, %": round(100 * counts[script] / len(records), 2),
        })

script_distribution = pd.DataFrame(script_rows).set_index(["split", "script"])
script_distribution

documents  share, %
split script                       
train latin          8097     62.28
      cyrillic       2799     21.53
      mixed          2103     16.18
      other             1      0.01
dev   latin           928     61.87
      cyrillic        345     23.00
      mixed           227     15.13
      other             0      0.00

### 1.5. Пустые записи и точные дубли

Различаем пустой текст и корректный текст без размеченных сущностей. Для дублей отдельно считаем повторяющиеся `hash`, группы одинаковых текстов внутри split и совпадения текстов между `train` и `dev`.

In [43]:
quality_rows = []
for split_name, records in splits.items():
    hash_counts = Counter(record["hash"] for record in records)
    text_counts = Counter(record["text"] for record in records)
    quality_rows.append({
        "split": split_name,
        "empty_texts": sum(not record["text"].strip() for record in records),
        "without_entities": sum(not record["entities"] for record in records),
        "duplicate_hash_groups": sum(count > 1 for count in hash_counts.values()),
        "duplicate_text_groups": sum(count > 1 for count in text_counts.values()),
        "extra_duplicate_documents": sum(
            count - 1 for count in text_counts.values() if count > 1
        ),
    })

quality_summary = pd.DataFrame(quality_rows).set_index("split")
train_texts = {record["text"] for record in train_records}
dev_texts = {record["text"] for record in dev_records}

display(quality_summary)
print(f"Одинаковых текстов между train и dev: {len(train_texts & dev_texts)}")

,empty_texts,without_entities,duplicate_hash_groups,duplicate_text_groups,extra_duplicate_documents
split,,,,,
train,0,2386,0,0,0
dev,0,275,0,1,1


Одинаковых текстов между train и dev: 0


### Вывод по пункту 1

`train` содержит 13 000 документов и 66 083 сущности, `dev` — 1 500 документов и 7 698 сущностей. Классы близки по размеру. Базовая проверка не обнаруживает некорректных меток, границ, пересечений и повторяющихся `hash`; точных совпадений текстов между `train` и `dev` также нет. Распределение длин тяжёлохвостое: кроме коротких сообщений встречаются тексты длиной в десятки тысяч символов, поэтому при обучении потребуется оконная токенизация. Данные содержат латиницу, кириллицу и существенную долю смешанных текстов — это необходимо учитывать при выборе модели и расширении корпуса.

## 2. Исследовательский анализ и стилистика данных

Раздел выполнен как воспроизводимый LLM-анализ **всех 14 500 документов** (`train` + `dev`) с `seed=42`. Оркестратор построил страты по split, сигнатуре классов сущностей, письменности, квантилю длины, числу сущностей, датоподобному суффиксу хеша и повторяющимся шаблонам. Исполнители получали только синтетический `item_id` и `text`; связь с `hash` и золотые `entities` им не передавались.

По явному условию повторного запуска первоначальная калибровка **не проводилась**. Ровно 20 исполнителей `gpt-5.6-sol` с `reasoning_effort=low` работали параллельно по таксономии v2.0.0; один передаваемый батч содержал не более 50 документов. Все документы получили хотя бы одну LLM-оценку, а 2 900 документов (20%) — две независимые оценки. Все первичные ответы, решения majority vote и отчёты находятся в `artifacts/style_analysis/`.

In [44]:
STYLE_ARTIFACT_DIR = PROJECT_DIR / "artifacts" / "style_analysis"

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]

taxonomy = load_json(STYLE_ARTIFACT_DIR / "taxonomy.json")
sampling_manifest = load_json(STYLE_ARTIFACT_DIR / "sampling_manifest.json")
agreement_report = load_json(STYLE_ARTIFACT_DIR / "agreement_report.json")
style_quality = load_json(STYLE_ARTIFACT_DIR / "quality_report.json")
style_predictions = load_jsonl(STYLE_ARTIFACT_DIR / "full_predictions.jsonl")
qa_review = load_jsonl(STYLE_ARTIFACT_DIR / "qa_review_sample.jsonl")
qa_conflicts = load_jsonl(STYLE_ARTIFACT_DIR / "qa_conflict_review.jsonl")

assert taxonomy["version"] == "2.0.0"
assert taxonomy["run_model"] == "gpt-5.6-sol"
assert taxonomy["run_reasoning_effort"] == "low"
assert taxonomy["calibration"] == "skipped_by_user_request"
assert sampling_manifest["seed"] == 42
assert sampling_manifest["population_documents"] == 14_500
assert sampling_manifest["sampling_design"]["main_double_annotated_documents"] == 2_900
assert sampling_manifest["sampling_design"]["batch_size_limit"] == 50
assert all(row["documents"] == 870 for row in sampling_manifest["assignment_summaries"].values())
assert len(style_predictions) == 14_500
assert len(qa_review) == 100 and all(row["manual_verdict"] for row in qa_review)
assert len(qa_conflicts) == 70 and all(row["review_status"] == "reviewed" for row in qa_conflicts)
assert style_quality["validation"]["all_valid"]

agreement_rows = []
for field in ("genre", "text_form", "register"):
        report = agreement_report["main_double_annotation"][field]
        agreement_rows.append({
            "field": field,
            "documents": report["documents_with_multiple_ratings"],
            "exact_agreement": report["exact_agreement"],
            "krippendorff_alpha": report["krippendorff_alpha_nominal"],
            "mean_jaccard": None,
        })
topics_agreement = agreement_report["main_double_annotation"]["topics"]
agreement_rows.append({
    "field": "topics",
    "documents": topics_agreement["rating_pairs"],
    "exact_agreement": topics_agreement["exact_agreement"],
    "krippendorff_alpha": None,
    "mean_jaccard": topics_agreement["mean_jaccard"],
})
agreement_summary = pd.DataFrame(agreement_rows)
agreement_summary

,field,documents,exact_agreement,krippendorff_alpha,mean_jaccard
0,genre,2900,0.877241,0.858156,NaN
1,text_form,2900,0.996552,0.936179,NaN
2,register,2900,0.868276,0.797104,NaN
3,topics,2900,0.723103,NaN,0.844575


### 2.1. Результаты LLM-разметки и контроль качества

Все 20 файлов исполнителей прошли структурную проверку: по 870 строк, суммарно 17 400 аннотаций для 14 500 документов. Каждый исполнитель обработал 18 последовательных батчей: 17 по 50 документов и последний из 20. Проверены JSONL, полнота и уникальность `item_id`, допустимые значения, диапазоны confidence и точное вхождение каждого `evidence` в исходный текст. Исходные `train.jsonl`/`dev.jsonl` не менялись.

Калибровочные метрики отсутствуют, поскольку этап был пропущен по условию эксперимента. На 2 900 дважды размеченных документах exact agreement составляет 87.7% для `genre`, 99.7% для `text_form` и 86.8% для `register`; Krippendorff's alpha — 0.858, 0.936 и 0.797 соответственно. Для мульти-лейбл `topics` exact agreement равен 72.3%, mean Jaccard — 0.845. Это сильное согласие в целом, но тематические границы и порядок дополнительных тем остаются заметным источником расхождений.

Конфликт хотя бы по одному полю обнаружен у 1 371 документа; 753 записи содержат неразрешённую ничью и помечены `needs_review`. Случайного tie-break нет. Среди разрешённых жанров чаще всего получены `comment_reply` (25.7%), `news_report` (19.0%), `reference_result` (13.3%) и `classified_listing` (8.8%); для регистра — `informal` (48.8%), `neutral` (29.8%) и `formal` (18.8%). Главный агент проверил фиксированную случайную сотню (`seed=42`): 83 записи приняты, у 17 найдено хотя бы одно содержательное расхождение. Дополнительно разобраны 70 конфликтных примеров: по 20 для `genre`, `topics` и `register`, а также все 10 конфликтов `text_form`. Аудит сохранён в `qa_review_sample.jsonl` и `qa_conflict_review.jsonl`; исходные ответы агентов не затирались.

### 2.2. Справочник `genre`

| `genre` | Что означает |
|---|---|
| `news_report` | Новость, репортаж, обзор или новостной дайджест |
| `official_announcement` | Сообщение госоргана, учреждения или пресс-службы от собственного имени |
| `social_post` | Самостоятельный пост или подпись к публикации в социальной сети/канале |
| `comment_reply` | Короткая реакция, мнение или ответ пользователя |
| `customer_review` | Оценка товара, заведения или услуги клиентом |
| `business_response` | Ответ организации клиенту, в том числе шаблонный |
| `advertisement` | Рекламная публикация, акция или общий призыв купить/заказать |
| `classified_listing` | Структурированное частное объявление с ценой, контактами или параметрами |
| `informational_educational` | Объяснение, инструкция, совет или учебный материал |
| `quiz_poll` | Викторина, тест, вопрос аудитории или голосование |
| `reference_result` | Ненарративные расписания, результаты, профили, списки и метаданные |
| `personal_message` | Поздравление, приглашение или личное сообщение |
| `mixed_composite` | В одной записи явно склеены тексты разных жанров или авторов |
| `other` | Фрагмент без достаточного контекста либо тип вне справочника |

### 2.3. Справочник `topics`

`topics` — мульти-лейбл поле: документу назначаются от одной до трёх тем, наиболее важная ставится первой.

| `topic` | Что включает |
|---|---|
| `politics_government` | Внутренняя политика, государственное управление, выборы |
| `international_relations_conflict` | Международные отношения, войны и конфликты |
| `law_crime_security` | Право, суды, преступления, коррупция, безопасность |
| `business_economy_finance` | Бизнес, экономика, рынки, банки и финансы |
| `construction_real_estate_urbanism` | Недвижимость, строительство и городская инфраструктура |
| `technology_telecom` | IT, цифровые сервисы, связь и телеком |
| `transport_automotive_aviation` | Автомобили, дороги, общественный транспорт и авиация |
| `sports_esports` | Спорт, соревнования и киберспорт |
| `entertainment_music_fandom` | Кино, музыка, шоу, знаменитости и фан-сообщества |
| `consumer_goods_services` | Еда, рестораны, магазины, товары и бытовые услуги |
| `health_medicine` | Здоровье, медицина и фармацевтика |
| `education_science` | Образование, обучение и наука |
| `employment_migration` | Работа, вакансии, трудовая и иная миграция |
| `society_social_issues` | Общественная жизнь и социальные проблемы |
| `religion_spirituality` | Религия и духовные практики |
| `culture_history_language` | Культура, история, литература и язык |
| `environment_agriculture` | Экология, погода, природные ресурсы и сельское хозяйство |
| `travel_geography` | Туризм, страны, города и достопримечательности |
| `personal_daily_life` | Отношения, семья и повседневная жизнь |
| `other` | Тема не определяется или не входит в справочник |

In [45]:
GENRE_LABELS = [
    "news_report", "official_announcement", "social_post",
    "comment_reply", "customer_review", "business_response",
    "advertisement", "classified_listing", "informational_educational",
    "quiz_poll", "reference_result", "personal_message",
    "mixed_composite", "other",
]

TOPIC_LABELS = [
    "politics_government", "international_relations_conflict",
    "law_crime_security", "business_economy_finance",
    "construction_real_estate_urbanism", "technology_telecom",
    "transport_automotive_aviation", "sports_esports",
    "entertainment_music_fandom", "consumer_goods_services",
    "health_medicine", "education_science", "employment_migration",
    "society_social_issues", "religion_spirituality",
    "culture_history_language", "environment_agriculture",
    "travel_geography", "personal_daily_life", "other",
]

TEXT_FORM_LABELS = ["native_text", "speech_transcript", "image_ocr"]
REGISTER_LABELS = ["formal", "neutral", "informal", "mixed"]

def resolved_distribution(field: str, labels: list[str]) -> pd.DataFrame:
    estimate = style_quality["weighted_estimates"][field]
    return pd.DataFrame([
        {"label": label,
         "resolved_documents": int(estimate["weighted_counts"][label]),
         "share_among_resolved, %": round(100 * estimate["share_among_resolved"][label], 2)}
        for label in labels
    ]).sort_values("resolved_documents", ascending=False)

genre_profile = resolved_distribution("genre", GENRE_LABELS)
register_profile = resolved_distribution("register", REGISTER_LABELS)
topic_profile = resolved_distribution("topics", TOPIC_LABELS)
display(genre_profile, register_profile, topic_profile)

,label,resolved_documents,"share_among_resolved, %"
3,comment_reply,3629,25.66
0,news_report,2682,18.96
10,reference_result,1886,13.33
7,classified_listing,1244,8.80
2,social_post,810,5.73
6,advertisement,787,5.56
1,official_announcement,707,5.00
11,personal_message,513,3.63
4,customer_review,487,3.44
5,business_response,411,2.91


,label,resolved_documents,"share_among_resolved, %"
2,informal,6892,48.82
1,neutral,4206,29.79
0,formal,2654,18.80
3,mixed,366,2.59


,label,resolved_documents,"share_among_resolved, %"
9,consumer_goods_services,2794,12.17
7,sports_esports,2131,9.28
0,politics_government,1602,6.98
3,business_economy_finance,1592,6.93
18,personal_daily_life,1498,6.52
1,international_relations_conflict,1490,6.49
8,entertainment_music_fandom,1450,6.32
4,construction_real_estate_urbanism,1203,5.24
2,law_crime_security,1138,4.96
6,transport_automotive_aviation,1114,4.85


### 2.4. Поле `text_form`

`text_form` описывает, **как текст был получен**, а не его жанр или тему. Здесь `native_text` означает изначально письменный текст, а не «текст на родном языке».

| `text_form` | Критерий | Документов с разрешённым majority vote |
|---|---|---:|
| `native_text` | Текст изначально набран/опубликован письменно; нет признаков OCR или расшифровки речи | 14 120 |
| `speech_transcript` | Расшифровка аудио/видео; характерны метки вида `Спикер 13:` | 223 |
| `image_ocr` | Текст извлечён из изображения; характерна секция `Текст на изображении:` | 147 |

Для 10 документов голоса по форме разделились и majority vote не дал результата. `native_text` означает лишь отсутствие наблюдаемых признаков OCR/транскрипта. Это наиболее стабильное поле, но без исходного медиа оно всё равно не доказывает способ получения текста.

In [46]:
text_form_estimate = style_quality["weighted_estimates"]["text_form"]
resolved_text_forms = sum(text_form_estimate["weighted_counts"].values())
text_form_profile = pd.DataFrame([
    {"text_form": label,
     "documents": int(text_form_estimate["weighted_counts"][label]),
     "share_among_resolved, %": round(100 * text_form_estimate["share_among_resolved"][label], 2)}
    for label in TEXT_FORM_LABELS
])
text_form_profile

,text_form,documents,"share_among_resolved, %"
0,native_text,14120,97.45
1,speech_transcript,223,1.54
2,image_ocr,147,1.01


### 2.5. Правила разметки и гипотеза об источниках

- В корпусе представлены новости, официальные публикации, самостоятельные социальные посты, комментарии, отзывы и ответы компаний, реклама, объявления, справочные результаты, учебные и личные тексты. Встречаются склейки исходной публикации с пользовательским комментарием.
- Маркеры `Telegram`, ссылки на новостные сайты, подписи пресс-служб и типичная структура заголовок–дата–текст поддерживают гипотезу об агрегации новостных сайтов и публичных каналов/соцсетей.
- Формулы `Официальный ответ на отзыв`, оценки еды/сервиса/атмосферы и названия заведений указывают на выгрузки площадок отзывов; цены, контакты, параметры товара/поездки — на доски объявлений и чаты услуг.
- Явные секции `Текст на изображении:` и реплики `Спикер N:` подтверждают наличие OCR- и ASR-подпотоков. Конкретный сервис распознавания по тексту определить нельзя.
- Датоподобный суффикс части `hash` и отличающийся профиль коротких хешей совместимы с несколькими конвейерами сбора, но это не доказательство конкретного источника.
- Поле `channel` не включено в заданную схему и не имеет допустимого словаря, поэтому соглашение по нему не выдумывалось: в отчёте оно отмечено как unavailable, а отдельно рассчитано соглашение по `text_form`.
- Согласие Sol-разметчиков высокое, однако ручная случайная сотня всё ещё дала 17% записей хотя бы с одной существенной проблемой, главным образом в темах и жанре. Поэтому агрегированные доли полезны для исследовательского профиля корпуса, но не являются готовым ground truth; для обучения лучше использовать первичные аннотации вместе с флагами конфликтов и ручными решениями.

## 3. Покрытие сущностей между train и dev

Сущностью считаем точную подстроку `text[start:end]`. Покрытие проверяем **внутри того же класса**: например, строка из `dev/NAME` считается покрытой только при наличии такой формы в `train/NAME`. Отдельно считаем долю покрытых упоминаний и долю покрытых уникальных форм.

Сравниваются три режима: исходная строка; Unicode-aware `casefold`; `NFKC + casefold` с приведением распространённых вариантов апострофа к ASCII `'`. Пробелы, окончания и орфографию не исправляем, нечёткий поиск и лемматизацию не используем.

In [47]:
APOSTROPHE_CHARACTERS = "’‘ʻʼʹ՚`´＇ꞌ"
APOSTROPHE_TRANSLATION = str.maketrans({char: "'" for char in APOSTROPHE_CHARACTERS})
ENTITY_LABELS = ["GEO", "NAME", "ORG"]
NORMALIZATION_MODES = ["exact", "casefold", "casefold+apostrophes"]


def normalize_entity_mention(mention: str, mode: str) -> str:
    if mode == "exact":
        return mention
    if mode == "casefold":
        return mention.casefold()
    if mode == "casefold+apostrophes":
        return (
            unicodedata.normalize("NFKC", mention)
            .casefold()
            .translate(APOSTROPHE_TRANSLATION)
        )
    raise ValueError(f"Неизвестный режим нормализации: {mode}")


def extract_entity_mentions(records: list[dict], split_name: str) -> pd.DataFrame:
    rows = []
    for record in records:
        text = record["text"]
        for entity in record["entities"]:
            mention = text[entity["start"]:entity["end"]]
            rows.append({
                "split": split_name,
                "hash": record["hash"],
                "label": entity["label"],
                "mention": mention,
            })
    return pd.DataFrame(rows)


train_entity_mentions = extract_entity_mentions(train_records, "train")
dev_entity_mentions = extract_entity_mentions(dev_records, "dev")
entity_mentions = pd.concat(
    [train_entity_mentions, dev_entity_mentions], ignore_index=True
)

mention_counts = (
    entity_mentions.groupby(["split", "label"], sort=False)
    .size()
    .rename("mentions")
    .to_frame()
)
mention_counts

mentions
split label          
train GEO       21445
      ORG       23420
      NAME      21218
dev   GEO        2720
      ORG        2659
      NAME       2319

In [48]:
coverage_rows = []
for mode in NORMALIZATION_MODES:
    mode_rows = []
    for label in ENTITY_LABELS:
        train_values = {
            normalize_entity_mention(value, mode)
            for value in train_entity_mentions.loc[
                train_entity_mentions["label"] == label, "mention"
            ]
        }
        dev_values = [
            normalize_entity_mention(value, mode)
            for value in dev_entity_mentions.loc[
                dev_entity_mentions["label"] == label, "mention"
            ]
        ]
        dev_unique = set(dev_values)
        covered_occurrences = sum(value in train_values for value in dev_values)
        covered_unique = len(dev_unique & train_values)
        mode_rows.append({
            "normalization": mode,
            "label": label,
            "dev_occurrences": len(dev_values),
            "covered_occurrences": covered_occurrences,
            "occurrence_coverage, %": round(100 * covered_occurrences / len(dev_values), 2),
            "dev_unique_forms": len(dev_unique),
            "covered_unique_forms": covered_unique,
            "unique_coverage, %": round(100 * covered_unique / len(dev_unique), 2),
            "unseen_unique_forms": len(dev_unique) - covered_unique,
        })

    totals = {
        column: sum(row[column] for row in mode_rows)
        for column in (
            "dev_occurrences", "covered_occurrences",
            "dev_unique_forms", "covered_unique_forms",
            "unseen_unique_forms",
        )
    }
    mode_rows.append({
        "normalization": mode,
        "label": "ALL",
        **totals,
        "occurrence_coverage, %": round(
            100 * totals["covered_occurrences"] / totals["dev_occurrences"], 2
        ),
        "unique_coverage, %": round(
            100 * totals["covered_unique_forms"] / totals["dev_unique_forms"], 2
        ),
    })
    coverage_rows.extend(mode_rows)

coverage_summary = pd.DataFrame(coverage_rows).set_index(["normalization", "label"])
coverage_summary

dev_occurrences  covered_occurrences  \
normalization        label                                         
exact                GEO               2720                 1918   
                     NAME              2319                  959   
                     ORG               2659                 1582   
                     ALL               7698                 4459   
casefold             GEO               2720                 1995   
                     NAME              2319                  995   
                     ORG               2659                 1641   
                     ALL               7698                 4631   
casefold+apostrophes GEO               2720                 2000   
                     NAME              2319                  996   
                     ORG               2659                 1649   
                     ALL               7698                 4645   

                            occurrence_coverage, %  dev_unique_forms  \
normalization        label                                             
exact                GEO                     70.51              1401   
                     NAME                    41.35              1524   
                     ORG                     59.50              1609   
                     ALL                     57.92              4534   
casefold             GEO                     73.35              1336   
                     NAME                    42.91              1488   
                     ORG                     61.71              1534   
                     ALL                     60.16              4358   
casefold+apostrophes GEO                     73.53              1325   
                     NAME                    42.95              1488   
                     ORG                     62.02              1525   
                     ALL                     60.34              4338   

                            covered_unique_forms  unique_coverage, %  \
normalization        label                                             
exact                GEO                     718               51.25   
                     NAME                    416               27.30   
                     ORG                     705               43.82   
                     ALL                    1839               40.56   
casefold             GEO                     717               53.67   
                     NAME                    416               27.96   
                     ORG                     695               45.31   
                     ALL                    1828               41.95   
casefold+apostrophes GEO                     711               53.66   
                     NAME                    417               28.02   
                     ORG                     693               45.44   
                     ALL                    1821               41.98   

                            unseen_unique_forms  
normalization        label                       
exact                GEO                    683  
                     NAME                  1108  
                     ORG                    904  
                     ALL                   2695  
casefold             GEO                    619  
                     NAME                  1072  
                     ORG                    839  
                     ALL                   2530  
casefold+apostrophes GEO                    614  
                     NAME                  1071  
                     ORG                    832  
                     ALL                   2517

### 3.1. Результаты покрытия

При точном сравнении `train` покрывает **4 459 из 7 698 упоминаний dev (57,92%)** и **1 839 из 4 534 уникальных label-aware форм (40,56%)**. После `casefold` покрытие упоминаний возрастает до 60,16%, после дополнительной унификации апострофов — до **60,34%**. Для уникальных форм итоговое покрытие равно **41,98%**. Основной прирост даёт регистр; разные апострофы добавляют ещё 14 покрытых упоминаний.

В наиболее строгом нормализованном режиме покрытие упоминаний составляет 73,53% для `GEO`, 62,02% для `ORG` и 42,95% для `NAME`. Новых уникальных форм в `dev`: **1 071 `NAME`**, **832 `ORG`** и **614 `GEO`**. Следовательно, класс `NAME` содержит и наибольшее абсолютное число новых форм, и наибольшую их долю — 71,98%.

In [49]:
from collections import defaultdict

CANONICAL_MODE = "casefold+apostrophes"
entity_mentions = entity_mentions.copy()
entity_mentions["canonical"] = entity_mentions["mention"].map(
    lambda value: normalize_entity_mention(value, CANONICAL_MODE)
)

frequency_profile_rows = []
frequent_rows = []
rare_rows = []
for label_index, label in enumerate(ENTITY_LABELS):
    label_rows = entity_mentions[entity_mentions["label"] == label]
    canonical_counts = Counter(label_rows["canonical"])
    variants = defaultdict(Counter)
    for canonical, mention in zip(label_rows["canonical"], label_rows["mention"]):
        variants[canonical][mention] += 1

    def representative(canonical: str) -> str:
        return sorted(
            variants[canonical].items(), key=lambda pair: (-pair[1], pair[0])
        )[0][0]

    hapax = sorted(
        canonical for canonical, count in canonical_counts.items() if count == 1
    )
    frequency_profile_rows.append({
        "label": label,
        "unique_forms": len(canonical_counts),
        "hapax_forms": len(hapax),
        "hapax_share, %": round(100 * len(hapax) / len(canonical_counts), 2),
    })

    for canonical, count in sorted(
        canonical_counts.items(), key=lambda pair: (-pair[1], pair[0])
    )[:10]:
        frequent_rows.append({
            "label": label,
            "mention": representative(canonical),
            "frequency": count,
        })

    rare_sample = (
        pd.Series(hapax, dtype="object")
        .sample(n=min(10, len(hapax)), random_state=42 + label_index)
        .sort_values()
    )
    for canonical in rare_sample:
        rare_rows.append({
            "label": label,
            "mention": representative(canonical),
            "frequency": 1,
        })

frequency_profile = pd.DataFrame(frequency_profile_rows).set_index("label")
frequent_entities = pd.DataFrame(frequent_rows).set_index(["label", "mention"])
rare_entity_examples = pd.DataFrame(rare_rows).set_index(["label", "mention"])
display(frequency_profile, frequent_entities, rare_entity_examples)

,unique_forms,hapax_forms,"hapax_share, %"
label,,,
GEO,7315,5091,69.60
NAME,11052,8669,78.44
ORG,9661,6836,70.76


frequency
label mention                            
GEO   Oʻzbekiston                     816
      AQSh                            383
      Oʻzbekistonda                   341
      Eron                            335
      Эрон                            332
      Toshkent                        328
      АҚШ                             294
      Ўзбекистон                      252
      Rossiya                         237
      Oʻzbekiston Respublikasi        190
NAME  Tae                             860
      Jungkook                        787
      Jk                              697
      Jimin                           367
      Murod Nazarov                   299
      Taehyung                        280
      Jin                             265
      Yoongi                          226
      Мурод Назаров                   188
      Shavkat Mirziyoyev              149
ORG   Oqtepa Lavash                   672
      Telegram                        621
      Instagram                       502
      Pepsi                           483
      Murad Buildings                 356
      YouTube                         328
      Facebook                        315
      Mobiuz                          269
      KFC                             193
      Arsenal                         125

frequency
label mention                                                      
GEO   Boyovut tumaniga                                            1
      Chattanooga                                                 1
      Dream City                                                  1
      Gʻafur Gʻulom                                               1
      Shayxontohur tumaniga                                       1
      Texasining                                                  1
      Toshkent Markaziy temiryoʻl vokzaliga                       1
      Xaqqilobod                                                  1
      кохирадан                                                   1
      Модера таврдан                                              1
NAME  Abdurasul Ahmedov                                           1
      Arzikulov Ogʻabek                                           1
      Daniel Felipe Martínez                                      1
      Ehud Olmert                                                 1
      Farhod Sadullayev                                           1
      Leonid Bozhkov                                              1
      Mamur Saidqosimov                                           1
      MUHAMMAD YUNUS Q'ORIga                                      1
      Сангаре                                                     1
      Шохмирзо Умаров                                             1
ORG   Bayer 04 Leverkusen                                         1
      Iqtisodiyot va moliya vazirligidan                          1
      Navoiy davlat universiteti                                  1
      Oqtepani vodnik filiali                                     1
      Samarqand viloyati favqulodda vaziyatlar boshqa...          1
      Siviglia                                                    1
      Yandex market                                               1
      ОТА ОНА ҲАҚИДА                                              1
      РИНЦ                                                        1
      芝奇皇家戟DDR4-3200                                              1

In [50]:
unseen_rows = []
for label in ENTITY_LABELS:
    train_canonical = set(
        train_entity_mentions.loc[
            train_entity_mentions["label"] == label, "mention"
        ].map(lambda value: normalize_entity_mention(value, CANONICAL_MODE))
    )
    label_dev = dev_entity_mentions[
        dev_entity_mentions["label"] == label
    ].copy()
    label_dev["canonical"] = label_dev["mention"].map(
        lambda value: normalize_entity_mention(value, CANONICAL_MODE)
    )
    label_dev = label_dev[~label_dev["canonical"].isin(train_canonical)]
    unseen_counts = Counter(label_dev["canonical"])
    unseen_variants = defaultdict(Counter)
    for canonical, mention in zip(label_dev["canonical"], label_dev["mention"]):
        unseen_variants[canonical][mention] += 1
    for canonical, count in sorted(
        unseen_counts.items(), key=lambda pair: (-pair[1], pair[0])
    )[:10]:
        mention = sorted(
            unseen_variants[canonical].items(),
            key=lambda pair: (-pair[1], pair[0]),
        )[0][0]
        unseen_rows.append({
            "label": label,
            "mention": mention,
            "dev_frequency": count,
        })

top_unseen_dev_entities = pd.DataFrame(unseen_rows).set_index(["label", "mention"])

ENTITY_COVERAGE_DIR = PROJECT_DIR / "artifacts" / "entity_coverage"
ENTITY_COVERAGE_DIR.mkdir(parents=True, exist_ok=True)
coverage_summary.reset_index().to_csv(
    ENTITY_COVERAGE_DIR / "coverage_summary.csv", index=False
)
frequency_profile.reset_index().to_csv(
    ENTITY_COVERAGE_DIR / "frequency_profile.csv", index=False
)
frequent_entities.reset_index().to_csv(
    ENTITY_COVERAGE_DIR / "frequent_entities.csv", index=False
)
rare_entity_examples.reset_index().to_csv(
    ENTITY_COVERAGE_DIR / "rare_entity_examples.csv", index=False
)
top_unseen_dev_entities.reset_index().to_csv(
    ENTITY_COVERAGE_DIR / "top_unseen_dev_entities.csv", index=False
)

top_unseen_dev_entities

dev_frequency
label mention                          
GEO   Ахси                           15
      MONGʻIT                        11
      Ахсикат                         6
      Maskat                          4
      Бакиров                         4
      Эски Ахси                       4
      Bektemir tumani                 3
      Jomboy tumaniga                 3
      Labiau                          3
      Samarqand tumaniga              3
NAME  Zou                            21
      Miso                           20
      Syan                           17
      Zane                           17
      Mirna                          11
      Heeseungni                      9
      Syanning                        8
      Rana                            7
      Marvarid                        6
      Natalya                         6
ORG   Metall Asia                    20
      Moskvich                       11
      Monako                         10
      Motorola                        7
      Vivo                            5
      FSHF                            4
      Renntech                        4
      UCI                             4
      Yandex Plus                     4
      Atletiko Nasonal                3

### Вывод по пункту 3

Словарь поверхностных форм из `train` недостаточен как единственный механизм распознавания: даже после безопасной нормализации он не покрывает 39,66% упоминаний и 58,02% уникальных форм из `dev`. Особенно велик разрыв для `NAME`. При этом словарь имеет длинный хвост: среди нормализованных форм всего корпуса ровно один раз встречаются 69,60% `GEO`, 78,44% `NAME` и 70,76% `ORG`.

Для модели это означает, что запоминание gazetteer из `train` будет полезным дополнительным сигналом, но не заменит контекстное распознавание и субсловную модель. Нормализация регистра обязательна; унификация узбекских апострофов даёт небольшой дополнительный прирост. Поскольку окончания и варианты транслитерации намеренно не склеивались, приведённая оценка измеряет перенос поверхностных форм, а не тождество реальных объектов.

## 4. Внешние NER-датасеты

Аудит выполнен **4 сентября 2026 года** по первичным карточкам и официальным файлам. Для сравнения оставлены четыре узбекских корпуса. Проверяются размер, язык и письменность, домен, фактический формат, BIO/BIOES, качество, лицензия и пересечения с текущими `train/dev`.

Оба XLSX с Mendeley были скачаны во временный каталог через официальный public API, их SHA-256 совпали с метаданными репозитория; исходные файлы в проект не добавлялись. Слова `gold`, `expert-validated` и показатели авторов ниже не считаются независимой гарантией качества: фактическая проверка действительно обнаружила ошибки и расхождения.


In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import json

EXTERNAL_AUDIT_DIR = PROJECT_DIR / "artifacts" / "external_ner_analysis"
EXTERNAL_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

external_sources = [
    {
        "priority": 1,
        "dataset": "Uzbek NER Gold (uznlp-uz)",
        "size": "4 176 предложений; 59 569 токенов; 9 250 spans",
        "language_script": "узбекский; нормализованная латиница",
        "domain": "смешанный; точный provenance нужно проверить",
        "source_format": "TSV: Sentence, TokenOrder, Token, NER_Tag, pos; BIO; один train",
        "labels": "PER, ORG, LOC + MISC/MONEY/NUMERIC/TEMPORAL/WORK",
        "license": "CC BY 4.0",
        "quality_risk": "gold по заявлению авторов; проверить границы, кавычки и Unicode-апострофы",
        "decision": "первый кандидат после ручного аудита",
        "primary_url": "https://huggingface.co/datasets/uznlp-uz/uzbek_NER",
        "file_url": "https://huggingface.co/datasets/uznlp-uz/uzbek_NER/blob/main/Uzbek_NER_Gold.tsv",
    },
    {
        "priority": 2,
        "dataset": "25000-UzNER-5Style (2026)",
        "size": "25 000 Sentence ID; 285 736 token rows",
        "language_script": "узбекский; все предложения в файле латинские",
        "domain": "5 стилей × 5 000; 304 предложения synthetic",
        "source_format": "XLSX: ID, Manba, Uslub, Sentence, Word, BIOES-Tag; split-поля нет",
        "labels": "21 тип; целевые PER/ORG/GPE/LOC",
        "license": "CC BY 4.0 на карточке; проверить права первичных источников",
        "quality_risk": "BIOES ошибок 0; exact-дублей 0; есть near-duplicates и семантический шум",
        "decision": "крупный кандидат после provenance-аудита и нового split",
        "primary_url": "https://data.mendeley.com/datasets/2sxv4xhv4c/1",
        "file_url": "https://data.mendeley.com/datasets/2sxv4xhv4c/1/files/a34b20fb-1903-45dc-b0a0-f8f04f99d64e",
    },
    {
        "priority": 3,
        "dataset": "Mengliev et al. Uzbek NER (2024)",
        "size": "карточка: 2 000/25 865; XLSX: 1 988 unique IDs/25 866 rows",
        "language_script": "узбекский; латиница, неоднородные апострофы",
        "domain": "IDs ≤154 — lex.uz/official; остальные в основном созданы авторами",
        "source_format": "XLSX: Sentence, Word, BIOES-Tag, English-version (translation)",
        "labels": "PER, ORG, LOC",
        "license": "данные CC BY 4.0; статья CC BY-NC 4.0",
        "quality_risk": "30 BIOES-ошибок; 12 IDs пропущены; 81 группа нормализованных дублей",
        "decision": "не добавлять без исправления разметки и дедупликации",
        "primary_url": "https://data.mendeley.com/datasets/xf7pyvhb2v/1",
        "file_url": "https://data.mendeley.com/datasets/xf7pyvhb2v/1/files/d0ff7897-9bbc-4159-964b-3a081b53f527",
    },
    {
        "priority": 4,
        "dataset": "WikiANN / PAN-X, config uz",
        "size": "1 000 train + 1 000 validation + 1 000 test",
        "language_script": "узбекский; письменность проверять по строкам",
        "domain": "Wikipedia",
        "source_format": "Parquet: tokens[], ner_tags[], langs[], spans[]; IOB2",
        "labels": "PER, ORG, LOC",
        "license": "UNKNOWN на карточке Hugging Face",
        "quality_risk": "автоматическая/слабая разметка; короткие энциклопедические фразы",
        "decision": "не включать до выяснения прав и очистки",
        "primary_url": "https://huggingface.co/datasets/unimelb-nlp/wikiann",
        "file_url": "https://huggingface.co/datasets/unimelb-nlp/wikiann/tree/main/uz",
    },
]

external_sources_df = pd.DataFrame(external_sources).sort_values("priority")
external_sources_df.to_csv(
    EXTERNAL_AUDIT_DIR / "dataset_comparison.csv", index=False, encoding="utf-8"
)
external_sources_df[
    ["priority", "dataset", "size", "domain", "source_format", "license", "decision"]
].style.hide(axis="index").set_properties(**{"text-align": "left"})


priority,dataset,size,domain,source_format,license,decision
1,Uzbek NER Gold (uznlp-uz),4 176 предложений; 59 569 токенов; 9 250 spans,смешанный; точный provenance нужно проверить,"TSV: Sentence, TokenOrder, Token, NER_Tag, pos; BIO; один train",CC BY 4.0,первый кандидат после ручного аудита
2,25000-UzNER-5Style (2026),25 000 Sentence ID; 285 736 token rows,5 стилей × 5 000; 304 предложения synthetic,"XLSX: ID, Manba, Uslub, Sentence, Word, BIOES-Tag; split-поля нет",CC BY 4.0 на карточке; проверить права первичных источников,крупный кандидат после provenance-аудита и нового split
3,Mengliev et al. Uzbek NER (2024),карточка: 2 000/25 865; XLSX: 1 988 unique IDs/25 866 rows,IDs ≤154 — lex.uz/official; остальные в основном созданы авторами,"XLSX: Sentence, Word, BIOES-Tag, English-version (translation)",данные CC BY 4.0; статья CC BY-NC 4.0,не добавлять без исправления разметки и дедупликации
4,"WikiANN / PAN-X, config uz",1 000 train + 1 000 validation + 1 000 test,Wikipedia,"Parquet: tokens[], ner_tags[], langs[], spans[]; IOB2",UNKNOWN на карточке Hugging Face,не включать до выяснения прав и очистки


### 4.1. Где данные нужно посмотреть самостоятельно

Техническую структуру и все строки XLSX я проверил автоматически. Вам остаётся человеческая проверка смысла и границ — именно её нельзя надёжно заменить статистикой. Открывайте данные здесь:

1. **Uzbek NER Gold** — [Data Studio, `train`](https://huggingface.co/datasets/uznlp-uz/uzbek_NER/viewer/default/train), [сырой `Uzbek_NER_Gold.tsv`](https://huggingface.co/datasets/uznlp-uz/uzbek_NER/blob/main/Uzbek_NER_Gold.tsv), [README](https://huggingface.co/datasets/uznlp-uz/uzbek_NER/blob/main/README.md). Просмотрите до 50 случайных предложений: многословные `ORG`, кавычки/дефисы, апострофы и переходы `B→I`.

2. **25000-UzNER-5Style** — [прямой просмотр XLSX](https://data.mendeley.com/datasets/2sxv4xhv4c/1/files/a34b20fb-1903-45dc-b0a0-f8f04f99d64e) или [страница датасета / Download All](https://data.mendeley.com/datasets/2sxv4xhv4c/1). Просмотрите **батч до 50 предложений отдельно для каждого стиля**, отдельно строки с `Manba = Sintetik augmentatsiya`. Проверьте сомнительные `GPE/LOC`, короткие фрагменты, near-duplicates и права на `Lex.uz`, `Kun.uz`, YouTube, литературу и учебные материалы. В XLSX нет split-колонки, несмотря на описание 20k/5k, поэтому готовый авторский split нельзя восстановить из поля файла.

3. **Mengliev et al. (2024)** — [прямой просмотр `courpusNER.xlsx`](https://data.mendeley.com/datasets/xf7pyvhb2v/1/files/d0ff7897-9bbc-4159-964b-3a081b53f527), [страница Mendeley](https://data.mendeley.com/datasets/xf7pyvhb2v/1) и [статья](https://pmc.ncbi.nlm.nih.gov/articles/PMC11732609/). Сначала посмотрите проблемные IDs `143, 857, 858, 912, 1136, 1605, 1630, 1723, 1846`; затем сравните диапазоны `1–154` и `155–2000`. Обратите внимание на generic-role как `PER`, разорванные `ORG/LOC`, шаблонные повторы и неоднородные апострофы.

4. **WikiANN Uzbek** — [Data Studio, config `uz`, train](https://huggingface.co/datasets/unimelb-nlp/wikiann/viewer/uz/train), затем `validation/test`, и [файлы `uz`](https://huggingface.co/datasets/unimelb-nlp/wikiann/tree/main/uz). Проверьте декодирование числовых tags, шум слабой разметки и короткие строки. На [карточке](https://huggingface.co/datasets/unimelb-nlp/wikiann) лицензия сейчас `unknown`: корпус нельзя считать готовым к обучению/распространению до юридического уточнения.

Для каждого кандидата зафиксируйте решение `accept/fix/drop` по просмотренным предложениям. Если в архиве нет annotation guideline или provenance, это тоже отдельный риск, а не повод угадывать правила.


In [3]:
mendeley_xlsx_audit_rows = [
    {
        "dataset": "Mengliev et al. 2024",
        "official_file": "courpusNER.xlsx",
        "sha256": "e4915c37a06cbae51c2657933f8024ddc473e178492ed2d3909737de3ec3b68b",
        "sheet_columns": "Sheet1: Sentence, Word, BIOES-Tag, English-version (translation)",
        "token_rows": 25866,
        "sentence_units": "1 990 contiguous groups; 1 988 unique IDs; 12 IDs missing",
        "script": "1 982 latin + 8 no-letter groups; 0 Cyrillic/mixed",
        "bioes": "30 errors in 24 groups",
        "duplicates": "81 normalized groups; 101 extra occurrences",
        "metadata_risk": "no source/style/split; IDs 192/193 are non-contiguous",
    },
    {
        "dataset": "25000-UzNER-5Style",
        "official_file": "25000-UzNER-5Style Corpus.xlsx",
        "sha256": "ae9d4f3f9113936749c070c6ea45eaf845e5c44baf0b8373e5e03ef1595bfc81",
        "sheet_columns": "Dataset_25000: ID, Manba, Uslub, Sentence, Word, BIOES-Tag",
        "token_rows": 285736,
        "sentence_units": "25 000 unique ordered IDs; no missing IDs",
        "script": "25 000 latin; 0 Cyrillic/mixed",
        "bioes": "0 errors in all 25 000 sentences",
        "duplicates": "0 exact normalized duplicates; near-duplicates still possible",
        "metadata_risk": "source/style present; no split or explicit synthetic flag",
    },
]
mendeley_xlsx_audit_df = pd.DataFrame(mendeley_xlsx_audit_rows)

source_profile_rows = [
    {"source": "Lex.uz", "style": "Rasmiy uslub", "sentences": 4995, "synthetic": False},
    {"source": "Sintetik augmentatsiya", "style": "Rasmiy uslub", "sentences": 5, "synthetic": True},
    {"source": "Youtube.com", "style": "Soʻzlashuv uslubi", "sentences": 4758, "synthetic": False},
    {"source": "Sintetik augmentatsiya", "style": "Soʻzlashuv uslubi", "sentences": 242, "synthetic": True},
    {"source": "Xalqaro konferensiyasi doirasidagi Yosh olimlarning konferensiyasi", "style": "Ilmiy uslub", "sentences": 3999, "synthetic": False},
    {"source": "Hoshimov konferensiya toʻplamlari", "style": "Ilmiy uslub", "sentences": 949, "synthetic": False},
    {"source": "Sintetik augmentatsiya", "style": "Ilmiy uslub", "sentences": 52, "synthetic": True},
    {"source": "Kun.uz", "style": "Publitsistik uslub", "sentences": 4995, "synthetic": False},
    {"source": "Sintetik augmentatsiya", "style": "Publitsistik uslub", "sentences": 5, "synthetic": True},
    {"source": "Ikki eshik orasi", "style": "Badiiy uslub", "sentences": 4000, "synthetic": False},
    {"source": "Kecha va kunduz", "style": "Badiiy uslub", "sentences": 1000, "synthetic": False},
]
source_profile_df = pd.DataFrame(source_profile_rows)

mendeley_entity_counts_rows = [
    {"dataset": "Mengliev 2024", "raw_label": "PER", "candidate_spans": 2365, "valid_spans": 2363},
    {"dataset": "Mengliev 2024", "raw_label": "ORG", "candidate_spans": 1105, "valid_spans": 1100},
    {"dataset": "Mengliev 2024", "raw_label": "LOC", "candidate_spans": 1374, "valid_spans": 1371},
    {"dataset": "25000-UzNER-5Style", "raw_label": "ORG", "candidate_spans": 6203, "valid_spans": 6203},
    {"dataset": "25000-UzNER-5Style", "raw_label": "LAW", "candidate_spans": 5039, "valid_spans": 5039},
    {"dataset": "25000-UzNER-5Style", "raw_label": "CARDINAL", "candidate_spans": 4670, "valid_spans": 4670},
    {"dataset": "25000-UzNER-5Style", "raw_label": "DATE", "candidate_spans": 4144, "valid_spans": 4144},
    {"dataset": "25000-UzNER-5Style", "raw_label": "PER", "candidate_spans": 3921, "valid_spans": 3921},
    {"dataset": "25000-UzNER-5Style", "raw_label": "GPE", "candidate_spans": 3509, "valid_spans": 3509},
    {"dataset": "25000-UzNER-5Style", "raw_label": "ORDINAL", "candidate_spans": 751, "valid_spans": 751},
    {"dataset": "25000-UzNER-5Style", "raw_label": "EXAM", "candidate_spans": 525, "valid_spans": 525},
    {"dataset": "25000-UzNER-5Style", "raw_label": "EVENT", "candidate_spans": 472, "valid_spans": 472},
    {"dataset": "25000-UzNER-5Style", "raw_label": "PRODUCT", "candidate_spans": 410, "valid_spans": 410},
    {"dataset": "25000-UzNER-5Style", "raw_label": "MISC", "candidate_spans": 401, "valid_spans": 401},
    {"dataset": "25000-UzNER-5Style", "raw_label": "QUANTITY", "candidate_spans": 318, "valid_spans": 318},
    {"dataset": "25000-UzNER-5Style", "raw_label": "PERCENT", "candidate_spans": 312, "valid_spans": 312},
    {"dataset": "25000-UzNER-5Style", "raw_label": "MONEY", "candidate_spans": 197, "valid_spans": 197},
    {"dataset": "25000-UzNER-5Style", "raw_label": "FAC", "candidate_spans": 149, "valid_spans": 149},
    {"dataset": "25000-UzNER-5Style", "raw_label": "WORK-OF-ART", "candidate_spans": 127, "valid_spans": 127},
    {"dataset": "25000-UzNER-5Style", "raw_label": "AGE", "candidate_spans": 51, "valid_spans": 51},
    {"dataset": "25000-UzNER-5Style", "raw_label": "NORP", "candidate_spans": 48, "valid_spans": 48},
    {"dataset": "25000-UzNER-5Style", "raw_label": "LOC", "candidate_spans": 29, "valid_spans": 29},
    {"dataset": "25000-UzNER-5Style", "raw_label": "LANGUAGE", "candidate_spans": 9, "valid_spans": 9},
    {"dataset": "25000-UzNER-5Style", "raw_label": "TIME", "candidate_spans": 5, "valid_spans": 5},
]
mendeley_entity_counts_df = pd.DataFrame(mendeley_entity_counts_rows)
mendeley_entity_counts_df["target_label"] = mendeley_entity_counts_df["raw_label"].map(
    {"PER": "NAME", "ORG": "ORG", "GPE": "GEO", "LOC": "GEO"}
)
mendeley_entity_counts_df["policy"] = mendeley_entity_counts_df["target_label"].notna().map(
    {True: "keep", False: "drop + report"}
)

target_span_summary = (
    mendeley_entity_counts_df.assign(
        kept=lambda frame: frame["valid_spans"].where(frame["target_label"].notna(), 0),
        dropped=lambda frame: frame["valid_spans"].where(frame["target_label"].isna(), 0),
    )
    .groupby("dataset", as_index=False)[["valid_spans", "kept", "dropped"]]
    .sum()
)

mendeley_xlsx_audit_df.to_csv(EXTERNAL_AUDIT_DIR / "mendeley_xlsx_audit.csv", index=False)
source_profile_df.to_csv(EXTERNAL_AUDIT_DIR / "mendeley_25k_source_profile.csv", index=False)
mendeley_entity_counts_df.to_csv(EXTERNAL_AUDIT_DIR / "mendeley_entity_counts.csv", index=False)

display(mendeley_xlsx_audit_df.style.hide(axis="index"))
display(target_span_summary.style.hide(axis="index"))
display(source_profile_df.style.hide(axis="index"))


dataset,official_file,sha256,sheet_columns,token_rows,sentence_units,script,bioes,duplicates,metadata_risk
Mengliev et al. 2024,courpusNER.xlsx,e4915c37a06cbae51c2657933f8024ddc473e178492ed2d3909737de3ec3b68b,"Sheet1: Sentence, Word, BIOES-Tag, English-version (translation)",25866,1 990 contiguous groups; 1 988 unique IDs; 12 IDs missing,1 982 latin + 8 no-letter groups; 0 Cyrillic/mixed,30 errors in 24 groups,81 normalized groups; 101 extra occurrences,no source/style/split; IDs 192/193 are non-contiguous
25000-UzNER-5Style,25000-UzNER-5Style Corpus.xlsx,ae9d4f3f9113936749c070c6ea45eaf845e5c44baf0b8373e5e03ef1595bfc81,"Dataset_25000: ID, Manba, Uslub, Sentence, Word, BIOES-Tag",285736,25 000 unique ordered IDs; no missing IDs,25 000 latin; 0 Cyrillic/mixed,0 errors in all 25 000 sentences,0 exact normalized duplicates; near-duplicates still possible,source/style present; no split or explicit synthetic flag


dataset,valid_spans,kept,dropped
25000-UzNER-5Style,31290,13662,17628
Mengliev 2024,4834,4834,0


source,style,sentences,synthetic
Lex.uz,Rasmiy uslub,4995,False
Sintetik augmentatsiya,Rasmiy uslub,5,True
Youtube.com,Soʻzlashuv uslubi,4758,False
Sintetik augmentatsiya,Soʻzlashuv uslubi,242,True
Xalqaro konferensiyasi doirasidagi Yosh olimlarning konferensiyasi,Ilmiy uslub,3999,False
Hoshimov konferensiya toʻplamlari,Ilmiy uslub,949,False
Sintetik augmentatsiya,Ilmiy uslub,52,True
Kun.uz,Publitsistik uslub,4995,False
Sintetik augmentatsiya,Publitsistik uslub,5,True
Ikki eshik orasi,Badiiy uslub,4000,False


In [4]:
format_comparison_rows = [
    {
        "dataset": "Текущий train/dev",
        "unit": "документ",
        "schema": "JSONL: hash, text, entities[{label,start,end}]",
        "coordinates": "Unicode code-point offsets [start, end)",
        "required_conversion": "нет",
    },
    {
        "dataset": "Uzbek NER Gold",
        "unit": "токен в строке TSV",
        "schema": "Sentence, TokenOrder, Token, NER_Tag, pos",
        "coordinates": "BIO tags; готовых char offsets нет",
        "required_conversion": "group → reconstruct/align → BIO spans → offsets",
    },
    {
        "dataset": "Mengliev 2024",
        "unit": "токен в строке XLSX",
        "schema": "Sentence ID, Word, BIOES-Tag, token translation",
        "coordinates": "BIOES; исходного sentence text/offsets нет",
        "required_conversion": "repair BIOES/IDs → group → reconstruct → offsets",
    },
    {
        "dataset": "25000-UzNER-5Style",
        "unit": "токен в строке XLSX",
        "schema": "ID, Manba, Uslub, Sentence ID, Word, BIOES-Tag",
        "coordinates": "BIOES; исходного sentence text/offsets нет",
        "required_conversion": "define detokenization → map 4 types → offsets → new split",
    },
    {
        "dataset": "WikiANN uz",
        "unit": "предложение",
        "schema": "tokens[], ner_tags[], langs[], spans[]",
        "coordinates": "IOB2 ids; готовых char offsets нет",
        "required_conversion": "decode ids → reconstruct → BIO spans → offsets",
    },
]
format_comparison_df = pd.DataFrame(format_comparison_rows)

label_mapping_rows = [
    {"external": "PER / PERSON", "target": "NAME", "policy": "перенести"},
    {"external": "ORG / ORGANIZATION", "target": "ORG", "policy": "перенести"},
    {"external": "LOC / LOCATION / GPE / GEO", "target": "GEO", "policy": "перенести после аудита guideline"},
    {
        "external": "MISC, MONEY, NUMERIC, TEMPORAL, WORK, DATE, LAW, PRODUCT, FAC, RANK, ...",
        "target": None,
        "policy": "исключить и обязательно посчитать; не сливать в ближайший класс",
    },
]
label_mapping_df = pd.DataFrame(label_mapping_rows)

format_comparison_df.to_csv(
    EXTERNAL_AUDIT_DIR / "format_comparison.csv", index=False, encoding="utf-8"
)
label_mapping_df.to_csv(
    EXTERNAL_AUDIT_DIR / "label_mapping.csv", index=False, encoding="utf-8"
)

display(format_comparison_df.style.hide(axis="index"))
display(label_mapping_df.style.hide(axis="index"))


dataset,unit,schema,coordinates,required_conversion
Текущий train/dev,документ,"JSONL: hash, text, entities[{label,start,end}]","Unicode code-point offsets [start, end)",нет
Uzbek NER Gold,токен в строке TSV,"Sentence, TokenOrder, Token, NER_Tag, pos",BIO tags; готовых char offsets нет,group → reconstruct/align → BIO spans → offsets
Mengliev 2024,токен в строке XLSX,"Sentence ID, Word, BIOES-Tag, token translation",BIOES; исходного sentence text/offsets нет,repair BIOES/IDs → group → reconstruct → offsets
25000-UzNER-5Style,токен в строке XLSX,"ID, Manba, Uslub, Sentence ID, Word, BIOES-Tag",BIOES; исходного sentence text/offsets нет,define detokenization → map 4 types → offsets → new split
WikiANN uz,предложение,"tokens[], ner_tags[], langs[], spans[]",IOB2 ids; готовых char offsets нет,decode ids → reconstruct → BIO spans → offsets


external,target,policy
PER / PERSON,NAME,перенести
ORG / ORGANIZATION,ORG,перенести
LOC / LOCATION / GPE / GEO,GEO,перенести после аудита guideline
"MISC, MONEY, NUMERIC, TEMPORAL, WORK, DATE, LAW, PRODUCT, FAC, RANK, ...",None,исключить и обязательно посчитать; не сливать в ближайший класс


### 4.2. Преобразование BIO/BIOES в текущий JSONL

Текущий формат использует полуинтервалы `[start, end)`: сущность извлекается как `text[start:end]`. Безопасный порядок преобразования:

1. сгруппировать токены одного предложения в исходном `TokenOrder`;
2. если источник хранит исходный текст и offsets — сохранить их; иначе детерминированно восстановить текст, одновременно накапливая позиции токенов;
3. декодировать числовые WikiANN tags через metadata конкретной версии, а не хардкодить ids;
4. объединить `B-X I-X ...` или `B-X I-X E-X`; `S-X`/`U-X` образует отдельный span;
5. применить отображение `PER→NAME`, `ORG→ORG`, `LOC/GPE→GEO`; прочие типы исключить **с отчётом количества**;
6. проверить для каждого span `text[start:end]`, отсутствие пустых/пересекающихся интервалов и допустимость последовательностей `I/E`;
7. только после offsets нормализовать копии строк для дедупликации. Менять Unicode, апострофы или пробелы до вычисления offsets нельзя.

Если исходного текста нет (например, WikiANN), соединение токенов пробелом меняет исходную типографику, но создаёт самосогласованный новый `text`. Повторный поиск токена с начала строки через `text.find` использовать нельзя: он ломается на повторяющихся словах.


In [5]:
TARGET_LABEL_MAP = {
    "PER": "NAME",
    "PERSON": "NAME",
    "ORG": "ORG",
    "ORGANIZATION": "ORG",
    "LOC": "GEO",
    "LOCATION": "GEO",
    "GPE": "GEO",
    "GEO": "GEO",
}


def _decode_ner_tag(tag, id2label=None) -> str:
    """Декодирует строковый tag или id из metadata конкретного датасета."""
    if isinstance(tag, int):
        if id2label is None:
            raise ValueError("Для числовых ner_tags требуется id2label из metadata")
        if isinstance(id2label, dict):
            value = id2label.get(tag, id2label.get(str(tag)))
        else:
            value = id2label[tag]
        if value is None:
            raise ValueError(f"Неизвестный tag id: {tag}")
        return str(value)
    return str(tag)


def _reconstruct_text_and_offsets(tokens, separator=" "):
    """Восстанавливает текст и offsets за один проход, без text.find."""
    parts, offsets = [], []
    cursor = 0
    for index, raw_token in enumerate(tokens):
        token = str(raw_token)
        if not token:
            raise ValueError(f"Пустой token на позиции {index}")
        if index:
            parts.append(separator)
            cursor += len(separator)
        start = cursor
        parts.append(token)
        cursor += len(token)
        offsets.append((start, cursor))
    return "".join(parts), offsets


def bioes_tokens_to_char_record(
    tokens,
    tags,
    *,
    source_id=None,
    id2label=None,
    label_map=TARGET_LABEL_MAP,
    separator=" ",
):
    """
    Конвертирует BIO/IOB2/BIOES/BILOU в текущую JSONL-схему.

    Невалидный I/E/L вызывает ошибку. Нетаргетные типы удаляются с явным
    отчётом по token-tags, а не молча превращаются в другой класс.
    """
    if len(tokens) != len(tags):
        raise ValueError(f"tokens={len(tokens)} и tags={len(tags)} имеют разную длину")

    text, token_offsets = _reconstruct_text_and_offsets(tokens, separator)
    entities = []
    active = None
    dropped_token_tags = Counter()

    def close_active():
        nonlocal active
        if active is not None:
            entities.append(active)
            active = None

    for index, (raw_tag, (start, end)) in enumerate(zip(tags, token_offsets)):
        tag = _decode_ner_tag(raw_tag, id2label).strip()
        if tag.upper() == "O":
            close_active()
            continue
        if "-" not in tag:
            raise ValueError(f"Некорректный tag {tag!r} на позиции {index}")

        prefix, raw_type = tag.split("-", 1)
        prefix, raw_type = prefix.upper(), raw_type.upper()
        if prefix not in {"B", "I", "E", "S", "L", "U"}:
            raise ValueError(f"Неизвестный префикс {prefix!r} на позиции {index}")

        target_label = label_map.get(raw_type)
        if target_label is None:
            close_active()
            dropped_token_tags[raw_type] += 1
            continue

        if prefix in {"S", "U"}:
            close_active()
            entities.append({"label": target_label, "start": start, "end": end})
        elif prefix == "B":
            close_active()
            active = {"label": target_label, "start": start, "end": end}
        elif prefix == "I":
            if active is None or active["label"] != target_label:
                raise ValueError(
                    f"Невалидный {tag!r} на позиции {index}: нет совместимого B-tag"
                )
            active["end"] = end
        else:  # E или L
            if active is None or active["label"] != target_label:
                raise ValueError(
                    f"Невалидный {tag!r} на позиции {index}: нет совместимого B-tag"
                )
            active["end"] = end
            close_active()

    close_active()
    entities.sort(key=lambda item: (item["start"], item["end"], item["label"]))

    for entity in entities:
        mention = text[entity["start"]:entity["end"]]
        if not mention:
            raise AssertionError(f"Пустая сущность: {entity}")
    for left, right in zip(entities, entities[1:]):
        if left["end"] > right["start"]:
            raise AssertionError(f"Пересекающиеся spans: {left}, {right}")

    if source_id is None:
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        source_id = f"external:{digest}"

    record = {"hash": str(source_id), "text": text, "entities": entities}
    report = {
        "input_tokens": len(tokens),
        "kept_entities": len(entities),
        "dropped_token_tags": dict(sorted(dropped_token_tags.items())),
        "reconstruction": "separator.join(tokens)",
            "separator": separator,
    }
    return record, report


In [6]:
example_tokens = [
    "Shavkat", "Mirziyoyev", "Toshkentga", "12", "sentabrda", "keldi", "."
]
example_tags = [
    "B-PER", "E-PER", "S-LOC", "B-DATE", "E-DATE", "O", "O"
]
example_record, example_report = bioes_tokens_to_char_record(
    example_tokens,
    example_tags,
    source_id="external-demo-1",
)

extracted_mentions = [
    (entity["label"], example_record["text"][entity["start"]:entity["end"]])
    for entity in example_record["entities"]
]
assert extracted_mentions == [
    ("NAME", "Shavkat Mirziyoyev"),
    ("GEO", "Toshkentga"),
]
assert example_report["dropped_token_tags"] == {"DATE": 2}

print(json.dumps(example_record, ensure_ascii=False, indent=2))
print("report:", example_report)


{
  "hash": "external-demo-1",
  "text": "Shavkat Mirziyoyev Toshkentga 12 sentabrda keldi .",
  "entities": [
    {
      "label": "NAME",
      "start": 0,
      "end": 18
    },
    {
      "label": "GEO",
      "start": 19,
      "end": 29
    }
  ]
}
report: {'input_tokens': 7, 'kept_entities': 2, 'dropped_token_tags': {'DATE': 2}, 'reconstruction': 'separator.join(tokens)', 'separator': ' '}


### 4.3. Пересечения с текущими train/dev

Exact-аудит выполнен для официальных файлов всех четырёх корпусов. Предложения сравнивались после `NFKC + casefold + унификация апострофов + схлопывание пробелов`; учитывались нормализованные строки длиной не менее 20 символов. Дополнительно проверено сравнение без пробелов перед пунктуацией — для обоих Mendeley XLSX результат также `0` совпадений предложений. Формы сущностей сравнивались только внутри соответствующего target-класса.

Это не доказательство отсутствия утечки: перед обучением нужны fuzzy/MinHash-поиск, дедупликация **до** формирования split и сохранение `source/version/original_id`. Совпадение одной entity surface form (например, названия города) само по себе не является дублем документа.


In [7]:
sentence_overlap_rows = [
    {
        "dataset": "Uzbek NER Gold",
        "published_or_actual_sentences": 4176,
        "comparable_unique_sentences": 4152,
        "exact_overlap_train": 0,
        "exact_overlap_dev": 0,
        "status": "официальный HF-файл",
    },
    {
        "dataset": "WikiANN uz",
        "published_or_actual_sentences": 3000,
        "comparable_unique_sentences": 989,
        "exact_overlap_train": 2,
        "exact_overlap_dev": 0,
        "status": "три официальных HF splits",
    },
    {
        "dataset": "Mengliev et al. 2024",
        "published_or_actual_sentences": 1988,
        "comparable_unique_sentences": 1872,
        "exact_overlap_train": 0,
        "exact_overlap_dev": 0,
        "status": "официальный XLSX; actual IDs вместо заявленных 2 000",
    },
    {
        "dataset": "25000-UzNER-5Style",
        "published_or_actual_sentences": 25000,
        "comparable_unique_sentences": 24905,
        "exact_overlap_train": 0,
        "exact_overlap_dev": 0,
        "status": "официальный XLSX",
    },
]
sentence_overlap_df = pd.DataFrame(sentence_overlap_rows)

entity_overlap_rows = [
    {"dataset": "Uzbek NER Gold", "label": "NAME", "external_unique": 531, "in_train": 57, "in_dev": 14},
    {"dataset": "Uzbek NER Gold", "label": "GEO", "external_unique": 734, "in_train": 341, "in_dev": 169},
    {"dataset": "Uzbek NER Gold", "label": "ORG", "external_unique": 850, "in_train": 200, "in_dev": 81},
    {"dataset": "WikiANN uz", "label": "NAME", "external_unique": 164, "in_train": 18, "in_dev": 4},
    {"dataset": "WikiANN uz", "label": "GEO", "external_unique": 964, "in_train": 32, "in_dev": 19},
    {"dataset": "WikiANN uz", "label": "ORG", "external_unique": 423, "in_train": 37, "in_dev": 14},
    {"dataset": "Mengliev et al. 2024", "label": "NAME", "external_unique": 357, "in_train": 12, "in_dev": 3},
    {"dataset": "Mengliev et al. 2024", "label": "GEO", "external_unique": 241, "in_train": 101, "in_dev": 57},
    {"dataset": "Mengliev et al. 2024", "label": "ORG", "external_unique": 225, "in_train": 51, "in_dev": 22},
    {"dataset": "25000-UzNER-5Style", "label": "NAME", "external_unique": 503, "in_train": 74, "in_dev": 24},
    {"dataset": "25000-UzNER-5Style", "label": "GEO", "external_unique": 236, "in_train": 137, "in_dev": 103},
    {"dataset": "25000-UzNER-5Style", "label": "ORG", "external_unique": 478, "in_train": 119, "in_dev": 59},
]
entity_overlap_df = pd.DataFrame(entity_overlap_rows)
entity_overlap_df["train_share_pct"] = (
    100 * entity_overlap_df["in_train"] / entity_overlap_df["external_unique"]
).round(1)
entity_overlap_df["dev_share_pct"] = (
    100 * entity_overlap_df["in_dev"] / entity_overlap_df["external_unique"]
).round(1)

sentence_overlap_df.to_csv(
    EXTERNAL_AUDIT_DIR / "sentence_overlap_audit.csv", index=False, encoding="utf-8"
)
entity_overlap_df.to_csv(
    EXTERNAL_AUDIT_DIR / "entity_surface_overlap.csv", index=False, encoding="utf-8"
)

manual_review_checklist = [
    {"dataset": row["dataset"], "url": row["file_url"], "status": "manual semantic review required"}
    for row in external_sources
]
pd.DataFrame(manual_review_checklist).to_csv(
    EXTERNAL_AUDIT_DIR / "manual_review_checklist.csv", index=False, encoding="utf-8"
)
audit_manifest = {
    "audited_at": "2026-09-04",
    "normalization": "NFKC + apostrophe unification + casefold + whitespace collapse",
    "sentence_min_normalized_chars": 20,
    "sources": external_sources,
    "mendeley_file_audit": mendeley_xlsx_audit_rows,
    "mendeley_25k_source_profile": source_profile_rows,
    "limitations": [
        "Exact sentence matching does not detect paraphrases, near-duplicates or partial reuse.",
        "Entity-surface overlap alone is not document leakage.",
        "The two XLSX files do not contain original raw sentence text or character offsets.",
        "Third-party source rights require separate provenance review.",
    ],
}
(EXTERNAL_AUDIT_DIR / "source_audit.json").write_text(
    json.dumps(audit_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(sentence_overlap_df.style.hide(axis="index"))
display(entity_overlap_df.style.hide(axis="index"))


dataset,published_or_actual_sentences,comparable_unique_sentences,exact_overlap_train,exact_overlap_dev,status
Uzbek NER Gold,4176,4152,0,0,официальный HF-файл
WikiANN uz,3000,989,2,0,три официальных HF splits
Mengliev et al. 2024,1988,1872,0,0,официальный XLSX; actual IDs вместо заявленных 2 000
25000-UzNER-5Style,25000,24905,0,0,официальный XLSX


dataset,label,external_unique,in_train,in_dev,train_share_pct,dev_share_pct
Uzbek NER Gold,NAME,531,57,14,10.700000,2.600000
Uzbek NER Gold,GEO,734,341,169,46.500000,23.000000
Uzbek NER Gold,ORG,850,200,81,23.500000,9.500000
WikiANN uz,NAME,164,18,4,11.000000,2.400000
WikiANN uz,GEO,964,32,19,3.300000,2.000000
WikiANN uz,ORG,423,37,14,8.700000,3.300000
Mengliev et al. 2024,NAME,357,12,3,3.400000,0.800000
Mengliev et al. 2024,GEO,241,101,57,41.900000,23.700000
Mengliev et al. 2024,ORG,225,51,22,22.700000,9.800000
25000-UzNER-5Style,NAME,503,74,24,14.700000,4.800000


### Вывод по пункту 4

1. **Первым проверять Uzbek NER Gold**: открытая лицензия, удобный viewer, 4 176 предложений и прямое отображение `PER/ORG/LOC`. После ручного аудита оставить только три целевых типа и конвертировать BIO.
2. **25000-UzNER-5Style — главный кандидат на масштабирование**, потому что все 25 000 последовательностей формально валидны и exact-дублей нет. Из 31 290 spans сохраняются `13 662` (`PER→NAME`, `ORG→ORG`, `GPE+LOC→GEO`), а `17 628` прочих нужно явно исключить. Новый split придётся строить самим: в XLSX split-поля нет. До использования проверить near-duplicates, семантику и права первичных источников.
3. **Mengliev et al. 2024 нельзя загружать как готовый gold**: в файле 1 988 уникальных IDs вместо 2 000, 30 BIOES-ошибок в 24 группах, разорванные IDs 192/193 и 81 группа нормализованных дублей. Сначала исправление и дедупликация; затем он может быть небольшим контролируемым дополнением.
4. **WikiANN uz не включать в основной train**, пока лицензия `unknown`; кроме того, разметка слабая и с текущим train найдены два точных совпадения предложений.

Для обоих XLSX исходный sentence text отсутствует: `Sentence` — только идентификатор. Поэтому offsets корректны лишь относительно заранее зафиксированного восстановленного текста, например `" ".join(tokens)`; он может содержать искусственные пробелы перед пунктуацией. Нормализацию Unicode/апострофов выполнять только после создания offsets.

Ни один внешний `test/validation` нельзя автоматически смешивать с обучением. Порядок интеграции: ручной `accept/fix/drop` → исправление BIO/BIOES → реконструкция и offsets → exact + fuzzy-дедупликация относительно текущего `dev` → новый train split с provenance.


## 5. Источники gazetteer

Gazetteer — словарь названий с типом сущности и ссылкой на реальный объект.
Проверены **GeoNames, OpenStreetMap, Wikidata и два официальных источника организаций Узбекистана**.
Для каждого ниже указаны формат, лицензия, охват, ограничения и польза. Дата проверки: **05.09.2026 (Москва)**.

Практическая часть: скачан полный GeoNames `UZ.zip`, построен словарь GEO и измерено
дополнительное покрытие `dev` относительно `train`. Остальные источники исследованы по документации
и страницам доступа: их объём и покрытие NER-корпуса не измерялись.
Все таблицы и исходный snapshot хранятся в `artifacts/gazetteer_analysis/`.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from scripts.eda.gazetteer_analysis import OUT as GAZETTEER_DIR, SOURCES as GAZETTEER_SOURCES

gazetteer_sources_df = pd.DataFrame(GAZETTEER_SOURCES)
display(gazetteer_sources_df[
    ["source", "labels", "format", "license", "coverage", "limits", "benefit", "status"]
].style.hide(axis="index").set_properties(**{"text-align": "left"}))


source,labels,format,license,coverage,limits,benefit,status
GeoNames UZ,GEO,"ZIP → UTF-8 TSV, 19 columns",CC BY 4.0 (gazetteer dump README),"Узбекистан: населённые пункты, административные и природные объекты; многоязычные aliases",Языки/историчность aliases отсутствуют в UZ.txt; для них нужен alternateNamesV2; вне-UZ география не покрывается,Первый словарь GEO; фактическое покрытие измерено ниже,"UZ.zip скачан, хеш зафиксирован; полный локальный аудит"
OpenStreetMap / Geofabrik UZ,GEO; ORG после проверки типа,OSM PBF; также SHP/GeoPackage; теги объектов,"ODbL 1.0; attribution, условия для производных БД","Населённые пункты, районы, улицы, локальные названия и POI; полнота зависит от картирования",Здание/бренд/оператор не равны GEO; дубли node/way/relation; name:uz/name:ru могут отсутствовать,"Дополнение локальной географии, отсутствующей в GeoNames","Формат и доступ проверены; PBF не загружался, покрытие не измерено"
Wikidata,NAME / ORG / GEO,Entity JSON/RDF; SPARQL JSON/CSV/TSV; labels + aliases + claims,CC0 для структурированных данных,"Известные люди, организации и места; uz/uz-cyrl/ru/en формы по наличию",Неполные labels и claims; QID не NER-класс; гражданство не равно языку; обычные люди представлены слабо,Многоязычные aliases и устойчивый QID; основной внешний кандидат для NAME,"Документация проверена; запросы подготовлены, полной выгрузки нет"
КТЯДР / registr.stat.uz,ORG,Веб-сервис/сведения по STIR (ИНН); bulk-файл/API не подтверждён,Открытая лицензия массового повторного использования не подтверждена,Юридические лица Узбекистана в государственном регистре,При проверке корень сайта перенаправляет в OneID; поиск по ID не даёт полного словаря; юридическое название отличается от бренда,Проверка официального названия и статуса известной организации,Регламент и сайт проверены; вход и массовая выгрузка не выполнялись
ЦБ Узбекистана: реестр банков,ORG,HTML-карточки + XLSX; языковые версии uz/oz/ru/en,Открытая лицензия на переиспользование XLSX не подтверждена,Коммерческие банки; другие финансовые реестры доступны отдельно,Только финансовый сектор; XLSX и HTML могут иметь разные даты; краткие бренды требуют сверки,Небольшой проверяемый отраслевой справочник ORG,Подтверждена XLSX-ссылка от 03.09.2026; строки файла и покрытие не измерены


### 5.1. Где брать данные и какие поля извлекать

**GeoNames → GEO.** [UZ.zip](https://download.geonames.org/export/dump/UZ.zip),
[описание схемы и лицензии](https://download.geonames.org/export/dump/readme.txt).
В UTF-8 TSV сохранять `geonameid`, `name`, `asciiname`, `alternatenames`,
`feature_class/code`, страну, административную иерархию и дату изменения.
Взяты классы `A/P` (административные единицы/населённые пункты), отдельно расширение
`H/T/V/L` (вода/рельеф/растительность/территории); `S/R` исключены из этого прототипа.
`alternatenames` не содержит языковых и исторических флагов. При их необходимости
использовать [alternateNamesV2](https://download.geonames.org/export/dump/alternateNamesV2.zip)
с фильтром по UZ geonameid. У самого gazetteer-дампа сейчас **CC BY 4.0**;
версия лицензии взята из скачанного README. `UZ.zip` не покрывает иностранные места в узбекских текстах.

**OpenStreetMap → GEO.** [Выгрузка Узбекистана Geofabrik](https://download.geofabrik.de/asia/uzbekistan.html)
доступна в PBF, SHP и GeoPackage. Для полного набора тегов предпочесть PBF.
Обрабатывать `place`, `boundary=administrative`, `waterway`, `natural` и именованные улицы,
согласовав границы GEO с нашей разметкой. Извлекать `name`, доступные `name:uz/name:ru`,
`official_name`, `short_name`, `alt_name`; `old_name` хранить отдельно.
[Семантика имён](https://wiki.openstreetmap.org/wiki/Key:name) важна для разделителей и вариантов.
Дедуплицировать по типу объекта + ID, затем проверять совпадающие node/way/relation.
`brand` и `operator` дают кандидатов ORG, но название здания или магазина нельзя автоматически
считать организацией. [Данные OSM — ODbL](https://www.openstreetmap.org/copyright):
учитывать attribution и условия распространения производных баз. Для полной выборки использовать extract;
[публичный Nominatim запрещает систематическое скачивание списков объектов](https://operations.osmfoundation.org/policies/nominatim/).

**Wikidata → NAME/ORG/GEO.** [Query Service](https://query.wikidata.org/),
[JSON/API/dumps](https://www.wikidata.org/wiki/Wikidata:Data_access),
[CC0 для структурированных данных](https://www.wikidata.org/wiki/Wikidata:Licensing).
Хранить QID, label и каждый alias отдельной строкой с языком, revision и происхождением.
Тип определять по `P31/P279`, связь с Узбекистаном — по гражданству `P27`, стране `P17`
или расположению штаб-квартиры `P159/P17`. `uz`, `uz-cyrl`, `ru`, `en` запрашиваются по наличию;
язык и письменность — разные поля. Фильтр гражданства исключает часть исторических/зарубежных лиц,
а отсутствие `P17` не означает, что организация не узбекская. Популярные люди представлены лучше
обычных участников комментариев: нельзя обещать полное покрытие NAME.

**КТЯДР → ORG.** [Реестр](https://registr.stat.uz/) и
[официальный регламент](https://stat.uz/img/xizmatlar/reglamentinteraktiv-xizmatlar.pdf)
подтверждают сервис сведений по STIR (ИНН). На проверке корень сайта перенаправляет в OneID.
Публичный bulk-export и лицензия повторного использования не подтверждены:
это источник проверки названий, а не готовая выгрузка всех компаний.
Сохранять юридическое имя, STIR, статус и дату; бренды и сокращения сверять отдельно.
Статистические таблицы с количеством предприятий не являются словарями названий.

**ЦБ Узбекистана → ORG.** [Реестр коммерческих банков](https://cbu.uz/uz/credit-organizations/banks/head-offices/)
содержит HTML-карточки и ссылку на XLSX от **03.09.2026**; есть языковые версии сайта.
Это точечный источник официальных имён банков, а не всех организаций страны.
Для сопоставления версий использовать стабильный идентификатор/номер лицензии, а не адрес.
[Другие финансовые реестры](https://cbu.uz/uz/credit-organizations/) расширяют отраслевой охват.
Наличие файла в открытом доступе не устанавливает открытую лицензию на его перераспространение;
условия для этого XLSX в просмотренной карточке не подтверждены.


In [2]:
# Шаблоны для первого просмотра, не результаты выгрузки.
# Каждый запрос возвращает <=50 QID. Затем labels/aliases получаются API-батчем <=50 ID.
WIKIDATA_PREFIXES = """PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
"""
wikidata_filters = {
    "NAME": "?item wdt:P31 wd:Q5; wdt:P27 wd:Q265 .",
    "ORG": """{ ?item wdt:P17 wd:Q265 . }
      UNION { ?item wdt:P159/wdt:P17 wd:Q265 . }
      ?item wdt:P31/wdt:P279* wd:Q43229 .""",
    "GEO": """?item wdt:P17 wd:Q265 .
      VALUES ?root { wd:Q486972 wd:Q56061 }
      ?item wdt:P31/wdt:P279* ?root .""",
}
gazetteer_queries = {
    label: WIKIDATA_PREFIXES + "SELECT DISTINCT ?item WHERE {\n"
    + pattern + "\n} ORDER BY ?item LIMIT 50\n"
    for label, pattern in wikidata_filters.items()
}
query_dir = GAZETTEER_DIR / "queries"
query_dir.mkdir(exist_ok=True)
for label, query in gazetteer_queries.items():
    (query_dir / f"wikidata_{label.lower()}.rq").write_text(query, encoding="utf-8")
    print(f"{label}: {query_dir / f'wikidata_{label.lower()}.rq'}")

display(pd.DataFrame([
    {"class": "NAME", "scope": "human (Q5), Uzbekistan citizenship (Q265)", "status": "template; not executed"},
    {"class": "ORG", "scope": "organization (Q43229); country or HQ country UZ", "status": "template; not executed"},
    {"class": "GEO", "scope": "human settlement (Q486972) or administrative entity (Q56061), UZ", "status": "template; not executed"},
]))


NAME: /home/mag/itmo/ai_hack/artifacts/gazetteer_analysis/queries/wikidata_name.rq
ORG: /home/mag/itmo/ai_hack/artifacts/gazetteer_analysis/queries/wikidata_org.rq
GEO: /home/mag/itmo/ai_hack/artifacts/gazetteer_analysis/queries/wikidata_geo.rq


,class,scope,status
0,NAME,"human (Q5), Uzbekistan citizenship (Q265)",template; not executed
1,ORG,organization (Q43229); country or HQ country UZ,template; not executed
2,GEO,human settlement (Q486972) or administrative e...,template; not executed


### 5.2. Формат словаря и правила использования

Одна строка — один вариант имени одного объекта. Общие поля:
`label, entity_id, name, normalized, language, script, alias_kind, source, license, snapshot/revision`.
Для географии полезны страна/иерархия/тип объекта; для ORG — официальный идентификатор и статус.
Одинаковое имя может вести к нескольким ID и классам: сохранять множество кандидатов.

Ключ сравнения: `NFKC → casefold → унификация апострофов → схлопывание пробелов`.
Исходное `name` сохраняется; текст документов и offsets не изменяются. Язык неизвестного alias
не угадывается по алфавиту. В данной проверке нет транслитерации, stemming и автоматического
удаления `-da/-ga/-dan/-ning`: для морфологии нужен отдельный эксперимент с контролем ложных совпадений.
Алиасы короче трёх символов и без букв исключены заранее.

Gazetteer использовать как признак для кандидатов модели: тип словаря, совпадение полного имени,
число подходящих ID, источник и длина совпадения. На этапе поиска в тексте нужны границы слов,
несколько длин совпадения и отображение нормализованных позиций на исходные offsets.
Этот раздел измеряет только совпадение **уже размеченных** полных упоминаний; matcher и модель здесь не обучаются.

Для Wikidata шаблоны ограничены первыми 50 ID и не являются оценкой объёма базы.
Для следующих страниц использовать keyset-пагинацию по последнему ID; результат не усекать молча.
Labels/aliases извлекать отдельным запросом к API с сохранением языков и revision.
Большие выгрузки делать через dumps, учитывая
[ограничения сервиса](https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/query_limits).
GEO-шаблон охватывает поселения и административные единицы, а не всю природную географию.


In [3]:
from scripts.eda.gazetteer_analysis import run as run_gazetteer_audit

# Полностью локальный пересчёт из сохранённого UZ.zip; сеть не используется.
gazetteer_summary = run_gazetteer_audit(download=False)
display(pd.DataFrame([{
    "objects_in_UZ_zip": gazetteer_summary["objects"],
    "selected_objects": gazetteer_summary["selected_objects"],
    "alias_rows": gazetteer_summary["aliases"],
    "normalized_keys": gazetteer_summary["normalized_aliases"],
    "ambiguous_keys": gazetteer_summary["ambiguous_alias_keys"],
    "snapshot_sha256": gazetteer_summary["snapshot"]["sha256"],
}]))
display(pd.DataFrame(gazetteer_summary["feature_classes"].items(), columns=["feature_class", "objects"]))
display(pd.DataFrame(gazetteer_summary["script_alias_rows"].items(), columns=["script", "alias_rows"]))
gazetteer_coverage_df = pd.DataFrame(gazetteer_summary["coverage"])
display(gazetteer_coverage_df.style.hide(axis="index"))


,objects_in_UZ_zip,selected_objects,alias_rows,normalized_keys,ambiguous_keys,snapshot_sha256
0,11157,10574,23900,17105,1793,2628d9bcdc46a15c2df320def81962351765b5a26f64e6...


,feature_class,objects
0,A,296
1,H,1665
2,L,80
3,P,8010
4,R,22
5,S,560
6,T,521
7,V,3


,script,alias_rows
0,cyrillic,1170
1,latin,21872
2,mixed,59
3,other,799


strategy,alias_keys,dev_mentions,matched_mentions,mention_coverage_pct,dev_unique,matched_unique,unique_coverage_pct,added_mentions_vs_train,added_unique_vs_train
train GEO,6699,2720,2001,73.570000,1325,712,53.740000,0,0
GeoNames A/P only,14060,2720,464,17.060000,1325,127,9.580000,31,25
GeoNames A/P/H/T/V/L,17105,2720,464,17.060000,1325,127,9.580000,31,25
train + GeoNames A/P,20338,2720,2032,74.710000,1325,737,55.620000,31,25
train + GeoNames A/P/H/T/V/L,23381,2720,2032,74.710000,1325,737,55.620000,31,25


### 5.3. Результат на текущем dev

В `UZ.zip` **11,157 объектов**; после отбора типов и имен —
**17,105 нормализованных ключей** для 10,574 объектов.
У 1,793 ключей несколько geonameid.
`script` в таблице различает наличие латиницы/кириллицы, `other` включает остальные письменности;
строки aliases не равны числу уникальных объектов или языков.

Словарь GEO из `train` покрывает **2001 из 2720 упоминаний dev
(73.57%)**. Добавление GeoNames даёт
**2032 (74.71%)**:
дополнительно **31 упоминание и 25 уникальных форм**.
Уникальное покрытие растёт с 53.74% до 55.62%.
GeoNames отдельно покрывает 464 упоминания (17,06%).

Все дополнительные совпадения уже достигаются с `A/P`; расширение `H/T/V/L` не дало новых
совпадений на этом dev. Это диагностический результат, а не основание удалять классы
по проверочной выборке. Небольшой прирост согласуется с ограничением одной страной,
окончаниями и вариантами записи; доля каждого фактора отдельно не измерялась.

Это **лексическое покрытие gold-упоминаний**, а не precision/recall/F1 NER или проверка
тождества географических объектов. Словарь создан независимо от dev; список новых форм ниже
предназначен для просмотра и не добавляется обратно в train. Snapshot современный:
для временного benchmark потребуется выгрузка не позднее границы времени теста.


In [4]:
gazetteer_review_df = pd.read_csv(GAZETTEER_DIR / "new_dev_geo_review.csv")
gazetteer_collisions_df = pd.read_csv(GAZETTEER_DIR / "cross_label_collisions.csv")
display(gazetteer_review_df.head(50).style.hide(axis="index"))
display(gazetteer_collisions_df.style.hide(axis="index"))


mention,dev_count,candidate_entities,geonames_ids,manual_decision
косон,3,1,geonames:1217007,pending
bektemir tumani,3,1,geonames:1536760,pending
qo'shrabot,2,2,geonames:12127802;geonames:1538239,pending
ohangaron,2,4,geonames:1513072;geonames:1514580;geonames:1514688;geonames:7577993,pending
angor tumani,1,1,geonames:1217872,pending
qaraqalpaqstan,1,1,geonames:453752,pending
sirg'ali tumani,1,1,geonames:1512848,pending
juma shahri,1,2,geonames:1217362;geonames:1346470,pending
qo'shrabot tumani,1,1,geonames:1513472,pending
navkat,1,2,geonames:1538536;geonames:1538570,pending


split,gold_label,mentions,matching_GEO_alias_mentions,matching_GEO_alias_unique,examples
train,NAME,21218,96,42,bobur; alisher navoiy; amir temur; go'zal; yulduz; zafar; salmon; oydin; malik; yunus
train,ORG,23420,288,59,paxtakor; bunyodkor; navbahor; andijon; бухоро; андижон; oqtepa; madrid; buxoro; mash'al
dev,NAME,2319,6,4,alisher navoiy; xassa; navoiy; marat
dev,ORG,2659,42,18,andijon; bunyodkor; paxtakor; g'allakor; madrid; buxoro; surxon; navbahor; xorazm; yuksalish


### Вывод по пункту 5

Начать с GeoNames как дополнительного GEO-признака, затем проверить Wikidata для многоязычных
NAME/ORG и известных мест. OSM полезен для локальных названий; официальные реестры — для
проверки юридических имён и отраслевых словарей. Измеренная польза GeoNames умеренная,
а польза остальных источников пока является гипотезой.

Уже есть межклассовые коллизии: GEO-ключи совпадают с **6 NAME и 42 ORG упоминаниями dev**.
Например, `Alisher Navoiy` может относиться к человеку или названному в его честь месту,
`Paxtakor/Bunyodkor/Andijon` — к организации либо месту. Это показывает необходимость контекста;
автоматическое присвоение GEO по одному совпадению создаёт ошибки.

**Где посмотреть самому:** откройте таблицы выше или
`artifacts/gazetteer_analysis/new_dev_geo_review.csv` (все 25 новых форм) и
`cross_label_collisions.csv`. Для объекта GeoNames откройте `https://www.geonames.org/<id>/`.
В `geonames_geo_aliases.csv` можно фильтровать `script=cyrillic/mixed` и неоднозначные имена.
Перед внесением ручных решений скопируйте review-таблицу: автоматический пересчёт пересоздаёт отчёт.
Онлайн-точки просмотра: [GeoNames](https://www.geonames.org/search.html?q=&country=UZ),
[OSM](https://www.openstreetmap.org/), [Wikidata](https://query.wikidata.org/),
[ЦБ](https://cbu.uz/uz/credit-organizations/banks/head-offices/), [КТЯДР](https://registr.stat.uz/).

**Повторить расчёт:** `uv run python scripts/eda/gazetteer_analysis.py`.
Для первоначальной загрузки при отсутствии snapshot: добавить `--download`.
Присоединять dev-формы к словарю запрещено методикой этого эксперимента; оценивать
NER-эффект затем на фиксированной проверочной выборке и отдельном итоговом test.


## 6. Генерация синтетических данных

## 7. Итоговые рекомендации